Environment: `velocity`

In [1]:
import scanpy
import anndata
import matplotlib
from matplotlib import pyplot
import hdf5plugin
import numpy
import scvelo
import seaborn
import pandas
import warnings
import cellrank
import gseapy
import decoupler
import IPython

In [2]:
# Read input file
working_directory = "RNA Sequencing Data/"

# adata = scanpy.read_h5ad(working_directory + "/velocity_sdevelo.h5ad")

# Or read saved anndata objects
working_directory = "RNA Sequencing Data/"
base_name = "cellrank_version_5"
adata = scanpy.read_h5ad(
    working_directory+f"/{base_name}_anndata.h5ad"
)
driver_df = pandas.read_csv(
    working_directory+f"{base_name}_driver_genes.csv",
    index_col=0
)
combined_driver_df = pandas.read_csv(
    working_directory+f"{base_name}_driver_genes_combined.csv",
    index_col=0
)

In [3]:
# Read ChIP-seq targets
base_name = "ChIP Sequencing Data/chipseq_targets"

hic2_target_scores = pandas.read_csv(f"{base_name}_hic2_scores.csv", index_col=0)
klf4_hic2_target_scores = pandas.read_csv(f"{base_name}_klf4_hic2_scores.csv", index_col=0)
klf4_control_target_scores = pandas.read_csv(f"{base_name}_klf4_control_scores.csv", index_col=0)

hic2_targets = numpy.loadtxt(f"{base_name}_hic2_targets.txt", dtype=numpy.dtypes.StrDType)
klf4_hic2_targets = numpy.loadtxt(f"{base_name}_klf4_hic2_targets.txt", dtype=numpy.dtypes.StrDType)
klf4_control_targets = numpy.loadtxt(f"{base_name}_klf4_control_targets.txt", dtype=numpy.dtypes.StrDType)
klf4_targets = numpy.intersect1d(klf4_hic2_targets, klf4_control_targets)
shared_targets = numpy.intersect1d(hic2_targets, klf4_targets)

In [4]:
# Load bulk differential expression results

bulk_de = pandas.read_csv("RNA Sequencing Data/bulk_differential_expression_combined.csv", index_col=0)
bulk_metadata = pandas.read_csv("RNA Sequencing Data/bulk_sequencing_metadata.csv", index_col=0)
samples = bulk_metadata.index.to_list()
reprogramming_columns = bulk_de.columns[bulk_de.columns.str.contains("Reprogramming")]
reprogramming_de = pandas.read_csv("RNA Sequencing Data/bulk_differential_expression_reprogramming.csv", index_col=0)
mef_de = pandas.read_csv("RNA Sequencing Data/bulk_differential_expression_mef.csv", index_col=0)
mesc_de = pandas.read_csv("RNA Sequencing Data/bulk_differential_expression_mesc.csv", index_col=0)
ipsc_de = pandas.read_csv("RNA Sequencing Data/bulk_differential_expression_ipsc.csv", index_col=0)

In [5]:
# Variable genes and Cellrank drivers
variable_genes = adata.var_names.to_numpy()
variable_in_bulk = numpy.intersect1d(variable_genes, bulk_de.index.to_numpy())
stem_cell_drivers = driver_df.sort_values("Day 12 Control_corr", ascending=True).head(100).index.to_numpy()
dead_end_drivers = driver_df.sort_values("Day 12 Control_corr", ascending=False).head(100).index.to_numpy()
differentially_expressed_genes = reprogramming_de.loc[variable_in_bulk].query("padj < 0.05").index.to_numpy()

In [6]:
# Load list of transcription factors

tf_df = pandas.read_csv("ChIP Sequencing Data/Mus_musculus_TF.txt", sep="\t")
tf_array = tf_df["Symbol"].to_numpy()

# All target transcription factors
target_tfs = numpy.intersect1d(shared_targets, tf_array)

# Target transcription factors for which Hic2 changes expression significantly
de_target_tfs = numpy.intersect1d(target_tfs, differentially_expressed_genes)

de_dead_end_drivers = numpy.intersect1d(de_target_tfs, dead_end_drivers)
de_stem_cell_drivers = numpy.intersect1d(de_target_tfs, stem_cell_drivers)

In [12]:
genes_of_interest = de_dead_end_drivers.tolist() + de_stem_cell_drivers.tolist() + ["Klf4", "Myc", "Hic2", "Cdh1"]

In [8]:
# Show all possible libraries
names = gseapy.get_library_name(organism="Mouse")
print(names)

['ARCHS4_Cell-lines', 'ARCHS4_IDG_Coexp', 'ARCHS4_Kinases_Coexp', 'ARCHS4_TFs_Coexp', 'ARCHS4_Tissues', 'Achilles_fitness_decrease', 'Achilles_fitness_increase', 'Aging_Perturbations_from_GEO_down', 'Aging_Perturbations_from_GEO_up', 'Allen_Brain_Atlas_10x_scRNA_2021', 'Allen_Brain_Atlas_down', 'Allen_Brain_Atlas_up', 'Azimuth_2023', 'Azimuth_Cell_Types_2021', 'BioCarta_2013', 'BioCarta_2015', 'BioCarta_2016', 'BioPlanet_2019', 'BioPlex_2017', 'CCLE_Proteomics_2020', 'CM4AI_U2OS_Protein_Localization_Assemblies', 'COMPARTMENTS_Curated_2025', 'COMPARTMENTS_Experimental_2025', 'CORUM', 'COVID-19_Related_Gene_Sets', 'COVID-19_Related_Gene_Sets_2021', 'Cancer_Cell_Line_Encyclopedia', 'Carcinogenome', 'CellMarker_2024', 'CellMarker_Augmented_2021', 'ChEA_2013', 'ChEA_2015', 'ChEA_2016', 'ChEA_2022', 'Chromosome_Location', 'Chromosome_Location_hg19', 'ClinVar_2019', 'ClinVar_2025', 'DGIdb_Drug_Targets_2024', 'DSigDB', 'Data_Acquisition_Method_Most_Popular_Genes', 'DepMap_CRISPR_GeneDependency

# Gene set enrichment for Cellrank drivers

## GO pathway

In [23]:
rank_data = driver_df[["Day 12 Control_corr"]].sort_values('Day 12 Control_corr', ascending=False).dropna()

# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets="GO_Biological_Process_2025",
    threads=16,
    min_size=5,
    max_size=1000,
    permutation_num=2000
)

2026-03-06 08:12:51,014 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [24]:
# Enriched in dead end

pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "ES", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]] # Top Day 12

,Term,ES,NES,NOM p-val,FWER p-val,FDR q-val
0,Regulation of Trans-Synaptic Signaling (GO:009...,0.7959,1.822757,0.003628,0.5035,0.768442
1,Epidermis Development (GO:0008544),0.649669,1.797313,0.002516,0.621,0.556069
2,Regulation of Glycolytic Process (GO:0006110),0.815407,1.78089,0.005006,0.697,0.473467
4,Proteolysis Involved in Protein Catabolic Proc...,0.740806,1.728133,0.013564,0.8955,0.714186
5,Antigen Processing and Presentation of Exogeno...,0.858955,1.708986,0.01171,0.9355,0.519213
6,Antigen Processing and Presentation of Peptide...,0.858955,1.708986,0.01171,0.9355,0.519213
7,Antigen Processing and Presentation of Exogeno...,0.858955,1.708986,0.01171,0.9355,0.519213
9,Negative Regulation of Myeloid Leukocyte Diffe...,0.758163,1.65506,0.016645,0.9925,0.843681
14,Positive Regulation of Purine Nucleotide Catab...,0.870543,1.606844,0.013564,0.999,1.0
15,Positive Regulation of Glycolytic Process (GO:...,0.870543,1.606844,0.013564,0.999,1.0


In [25]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
3,Chromatin Organization (GO:0006325),-1.734612,0.000799,0.5315,1.0
8,Protein Polyubiquitination (GO:0000209),-1.693375,0.005907,0.7375,1.0
10,DNA Metabolic Process (GO:0006259),-1.646768,0.010779,0.919,1.0
11,Transcription by RNA Polymerase II (GO:0006366),-1.640818,0.003416,0.935,1.0
12,Male Meiotic Nuclear Division (GO:0007140),-1.618388,0.001753,0.9675,1.0
13,Transcription Initiation-Coupled Chromatin Rem...,-1.614653,0.005872,0.973,1.0
16,Regulation of Insulin Secretion (GO:0050796),-1.604333,0.012701,0.981,1.0
17,Visual System Development (GO:0150063),-1.596589,0.009378,0.986,1.0
21,Regulation of Viral Genome Replication (GO:004...,-1.569126,0.022709,0.996,1.0
26,Regulation of Cholesterol Efflux (GO:0010874),-1.540434,0.011844,1.0,1.0


## GO molecular function

In [32]:
# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets="GO_Molecular_Function_2025",
    threads=16,
    min_size=5,
    max_size=1000,
    permutation_num=2000
)

2026-03-06 08:13:22,269 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [33]:
# Enriched in dead end

pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "ES", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]] # Top Day 12

,Term,ES,NES,NOM p-val,FWER p-val,FDR q-val
0,Chemokine Receptor Binding (GO:0042379),0.779504,1.874218,0.002421,0.052,0.064018
2,Exopeptidase Activity (GO:0008238),0.804972,1.65049,0.015837,0.5675,0.55066
3,Cysteine-Type Endopeptidase Activity (GO:0004197),0.67699,1.640115,0.025094,0.6055,0.408157
5,Chemokine Activity (GO:0008009),0.702967,1.607676,0.028605,0.728,0.424388
7,Cysteine-Type Peptidase Activity (GO:0008234),0.612025,1.577205,0.030713,0.8235,0.453548
8,Carboxypeptidase Activity (GO:0004180),0.763424,1.572703,0.03815,0.8345,0.394232
9,Monoatomic Cation Channel Activity (GO:0005261),0.665037,1.566582,0.033816,0.8515,0.355894
12,GTP Binding (GO:0005525),0.531162,1.545743,0.025538,0.8965,0.367965
13,Phosphatase Binding (GO:0019902),0.673177,1.489162,0.060932,0.964,0.500808
14,Guanyl Ribonucleotide Binding (GO:0032561),0.485501,1.478988,0.02642,0.972,0.481325


In [34]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
1,mRNA Binding (GO:0003729),-1.693617,0.001637,0.205,0.249014
4,Iron Ion Binding (GO:0005506),-1.625518,0.004156,0.4485,0.330839
6,2-Oxoglutarate-Dependent Dioxygenase Activity ...,-1.585645,0.008313,0.626,0.367861
10,Methylated Histone Binding (GO:0035064),-1.555595,0.006114,0.752,0.317115
11,Methylation-Dependent Protein Binding (GO:0140...,-1.555595,0.006114,0.752,0.317115
16,Protein Kinase C Binding (GO:0005080),-1.415454,0.062609,0.994,1.0
17,Guanyl-Nucleotide Exchange Factor Activity (GO...,-1.411386,0.072908,0.995,0.927603
20,Ubiquitin Binding (GO:0043130),-1.367428,0.09777,1.0,1.0
21,Single-Stranded DNA Binding (GO:0003697),-1.359997,0.10087,1.0,1.0
22,Protein Serine/Threonine Kinase Inhibitor Acti...,-1.33747,0.119167,1.0,1.0


## MSigDB_Hallmark_2020

In [35]:
rank_data = driver_df[["Day 12 Control_corr"]].sort_values('Day 12 Control_corr', ascending=False).dropna()

# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets='MSigDB_Hallmark_2020', # Or other libraries
    threads=16,
    min_size=5,
    max_size=1000,
)

2026-03-06 08:13:34,939 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [36]:
# Enriched in dead end

pre_res.res2d.sort_values('NES', ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]] # Top Day 12

,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,p53 Pathway,1.631392,0.0,0.189,0.392934
2,Estrogen Response Early,1.565239,0.011364,0.304,0.34522
3,Cholesterol Homeostasis,1.479324,0.050938,0.477,0.424119
5,Androgen Response,1.442451,0.050378,0.57,0.403225
9,Estrogen Response Late,1.382278,0.030812,0.697,0.469275
11,Allograft Rejection,1.355872,0.0625,0.741,0.456552
12,Unfolded Protein Response,1.322898,0.144608,0.805,0.466175
14,mTORC1 Signaling,1.275806,0.086842,0.878,0.530695
15,IL-2/STAT5 Signaling,1.268076,0.077135,0.888,0.493558
16,Wnt-beta Catenin Signaling,1.255005,0.205811,0.904,0.476947


In [37]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
1,G2-M Checkpoint,-1.590507,0.004673,0.218,0.121152
4,Spermatogenesis,-1.457728,0.04886,0.605,0.246197
6,Oxidative Phosphorylation,-1.394653,0.082759,0.789,0.275877
7,DNA Repair,-1.39212,0.076014,0.801,0.21226
8,Mitotic Spindle,-1.384432,0.062405,0.82,0.180998
10,E2F Targets,-1.358079,0.068293,0.876,0.182945
13,Interferon Alpha Response,-1.298227,0.096367,0.957,0.237578
25,Adipogenesis,-1.040436,0.381241,1.0,0.751119
30,Bile Acid Metabolism,-0.933019,0.54766,1.0,0.959
31,Interferon Gamma Response,-0.85101,0.777612,1.0,1.0


## Cell types

In [38]:
# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets='PanglaoDB_Augmented_2021', # Or other libraries
    threads=16,
    min_size=5,
    max_size=1000,
)

# Enriched in dead end

pre_res.res2d.sort_values('NES', ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]] # Top Day 12

2026-03-06 08:13:47,444 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,Salivary Mucous Cells,2.237623,0.0,0.0,0.0
1,Gastric Chief Cells,2.219473,0.0,0.0,0.0
2,Mammary Epithelial Cells,2.05964,0.0,0.002,0.000885
3,Cholangiocytes,2.052431,0.0,0.003,0.000995
4,Luminal Epithelial Cells,2.035843,0.0,0.004,0.001062
5,Keratinocytes,1.990003,0.0,0.007,0.00177
6,Foveolar Cells,1.956646,0.002513,0.016,0.003413
8,Sebocytes,1.933107,0.0,0.025,0.005143
9,Microfold Cells,1.927234,0.0,0.025,0.004571
10,Urothelial Cells,1.888582,0.0,0.041,0.006238


In [39]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
7,Pluripotent Stem Cells,-1.93891,0.0,0.007,0.005691
20,Epiblast Cells,-1.709858,0.001538,0.183,0.099181
23,Gamma Delta T Cells,-1.579801,0.010292,0.601,0.321931
24,Embryonic Stem Cells,-1.572849,0.001508,0.625,0.258317
26,Oxyphil Cells,-1.499128,0.039669,0.847,0.430867
29,Adrenergic Neurons,-1.423953,0.060403,0.958,0.679496
30,Myocytes,-1.40701,0.060559,0.971,0.66767
31,Ependymal Cells,-1.40181,0.095076,0.974,0.609209
34,Osteocytes,-1.367737,0.044515,0.986,0.699323
36,Microglia,-1.349372,0.07489,0.994,0.711012


Epithelial markers are strongly enriched among the control drivers

## ChipSeq targets, ChEA

In [40]:
# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets='ChEA_2022', # Or other libraries
    threads=16,
    min_size=5,
    max_size=1000
)

2026-03-06 08:14:06,477 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [41]:
# Enriched in dead end

pre_res.res2d.sort_values('NES', ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]] # Top Day 12

,Term,NES,NOM p-val,FWER p-val,FDR q-val
26,JARID2 20075857 ChIP-Seq MESCs Mouse,1.699716,0.0,0.282,0.44995
28,RARG 19884340 ChIP-ChIP MEFs Mouse,1.679999,0.0,0.33,0.271608
31,SUZ12 18974828 ChIP-Seq MESCs Mouse,1.661903,0.0,0.38,0.211321
39,MTF2 20144788 ChIP-Seq MESCs Mouse,1.62774,0.0,0.482,0.219303
41,SUZ12 27294783 Chip-Seq ESCs Mouse,1.624151,0.0,0.493,0.183005
52,TP63 17297297 ChIP-ChIP HaCaT Human,1.538354,0.033505,0.734,0.299546
55,SUZ12 20075857 ChIP-Seq MESCs Mouse,1.523259,0.0,0.78,0.290964
56,TP63 30713093 ChIP-Seq Epithelial Human Tongue...,1.508663,0.0,0.818,0.285
57,ESR1 26153859 ChIP-Seq MCF-7 Human BreastCancer,1.506626,0.0,0.821,0.256414
60,PITX1 30713093 ChIP-Seq Epithelial Human Tongu...,1.499521,0.06506,0.834,0.244258


In [42]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,FOXM1 23109430 ChIP-Seq U2OS Human,-2.102492,0.0,0.0,0.0
1,NACC1 18358816 ChIP-ChIP MESCs Mouse,-2.041287,0.0,0.0,0.0
2,SMAD1 18555785 ChIP-Seq MESCs Mouse,-2.037721,0.0,0.0,0.0
3,MYBL2 22936984 ChIP-ChIP MESCs Mouse,-2.036544,0.0,0.0,0.0
4,TCF3 18692474 ChIP-Seq MESCs Mouse,-2.006263,0.0,0.0,0.0
5,NANOG 18700969 ChIP-ChIP MESCs Mouse,-1.983808,0.0,0.002,0.000301
6,SOX2 18692474 ChIP-Seq MESCs Mouse,-1.938704,0.0,0.003,0.000387
7,POU5F1 18700969 ChIP-ChIP MESCs Mouse,-1.934232,0.0,0.005,0.000564
8,POU5F1 18358816 ChIP-ChIP MESCs Mouse,-1.925726,0.0,0.006,0.000601
9,NANOG 18555785 ChIP-Seq MESCs Mouse,-1.903279,0.0,0.008,0.000722


## Enriched targets from ChipSeq, Ensembl

In [43]:
# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets='ENCODE_TF_ChIP-seq_2015', # Or other libraries
    threads=16,
    min_size=5,
    max_size=1000
)

2026-03-06 08:14:40,109 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [44]:
# Enriched in dead end

pre_res.res2d.sort_values('NES', ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]] # Top Day 12

,Term,NES,NOM p-val,FWER p-val,FDR q-val
1,FOSL1 C2C12 mm9,1.934997,0.0,0.025,0.028715
13,SMARCC1 HeLa-S3 hg19,1.698293,0.0,0.284,0.203212
22,ZEB1 HepG2 hg19,1.601671,0.0125,0.526,0.332061
23,ATF3 K562 hg19,1.59631,0.003086,0.54,0.262298
27,TAL1 G1E-ER4 mm9,1.566733,0.003077,0.621,0.267047
28,EP300 ECC-1 hg19,1.566008,0.005865,0.622,0.223828
36,SMARCC2 HeLa-S3 hg19,1.523065,0.019886,0.755,0.285886
48,MAX myocyte mm9,1.466098,0.015625,0.88,0.411809
53,JUND GM12878 hg19,1.446921,0.026738,0.908,0.429985
55,RAD21 ECC-1 hg19,1.444632,0.006472,0.91,0.395049


In [45]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,E2F4 MEL cell line mm9,-2.08111,0.0,0.0,0.0
2,FOXM1 ECC-1 hg19,-1.842079,0.0,0.031,0.015764
3,IRF3 GM12878 hg19,-1.827508,0.0,0.04,0.013376
4,E2F4 CH12.LX mm9,-1.824025,0.0,0.043,0.010987
5,E2F4 HeLa-S3 hg19,-1.772204,0.0,0.095,0.02121
6,FOXM1 MCF-7 hg19,-1.742919,0.004717,0.147,0.028185
7,SP2 K562 hg19,-1.725061,0.0,0.187,0.031529
8,NFYA GM12878 hg19,-1.72164,0.0,0.193,0.028782
9,NFYA K562 hg19,-1.719114,0.0,0.198,0.026221
10,NANOG H1-hESC hg19,-1.716418,0.003378,0.205,0.02465


## TF Perturbation followed by expression

In [46]:
# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets="TF_Perturbations_Followed_by_Expression",
    threads=16,
    min_size=5,
    max_size=1000
)

2026-03-06 08:14:52,738 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


In [47]:
# Enriched in dead end

pre_res.res2d.sort_values('NES', ascending=False).head(30) # Top Day 12

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
20,prerank,HSF1 KO MOUSE GSE41005 CREEDSID GENE 2143 DOWN,0.691633,1.923269,0.00266,0.123783,0.107,9/20,5.66%,Lad1;Ccnd2;Pla2g12a;Fabp5;Tuba4a;Ifrd1;Myc;Atf...
22,prerank,IRF6 KO MOUSE GSE5800 CREEDSID GENE 153 UP,0.53671,1.914283,0.0,0.068328,0.118,29/70,15.42%,Lgals7;S100a16;Prr13;Lad1;Dsp;Pkp1;Gsto1;Perp;...
23,prerank,PPARA KO MOUSE GSE6864 CREEDSID GENE 981 UP,0.532728,1.900752,0.0,0.060076,0.149,31/67,17.28%,Ctsz;Prr13;Lad1;Gsto1;Ifi30;Cdkn1a;Krt8;Dgat2;...
28,prerank,PPARA KO MOUSE GSE6864 CREEDSID GENE 1371 DOWN,0.520718,1.860592,0.0,0.072785,0.226,31/68,17.28%,Ctsz;Prr13;Lad1;Gsto1;Ifi30;Cdkn1a;Krt8;Dgat2;...
40,prerank,PPARA DEFICIENCY MOUSE GSE6864 CREEDSID GENE 6...,0.511269,1.812715,0.0,0.108533,0.366,30/67,17.28%,Ctsz;Prr13;Lad1;Dsp;Gsto1;Krt8;Sat1;Tuba4a;Spi...
41,prerank,MYC KD HUMAN GSE22139 CREEDSID GENE 712 DOWN,0.633966,1.810695,0.002597,0.0916,0.369,13/22,13.87%,Ccnd2;Fabp5;Ifrd1;Myc;Tmem147;Zfas1;Snhg1;Ppa1...
44,prerank,OVOL2 OE MOUSE GSE55074 CREEDSID GENE 2603 DOWN,0.522185,1.793472,0.0,0.09549,0.433,26/56,14.22%,Lgals7;S100a16;Dsp;Ccnd2;Pkp1;Gsto1;Perp;Fabp5...
45,prerank,PPARG OE MOUSE GSE2192 CREEDSID GENE 863 UP,0.527031,1.793181,0.0,0.083801,0.435,21/48,13.87%,Nupr1;Pla2g12a;Fabp5;Cdkn1a;Mgst3;Tob1;Tmem147...
49,prerank,MYC OE MOUSE GSE55272 CREEDSID GENE 1822 DOWN,0.53103,1.781451,0.0,0.083843,0.465,13/50,6.01%,Lgals7;Dsp;Mal2;Perp;Ly6g6c;Ovol1;Anxa8;Krt17;...
51,prerank,YY1 KD MOUSE GSE31784 CREEDSID GENE 1333 UP,0.549218,1.778067,0.0,0.078132,0.476,16/40,14.17%,Lad1;Gsto1;Ly6g6c;Dgat2;Tubb3;Gpx2;Gatm;Pcolce...


Many proteins related to Ovol1 appear, including Myc, Ovol2 and Ovol1 itself. Directions are inconsistent.

In [48]:
# Enriched in iPSC path
pre_res.res2d.sort_values('NES', ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]  # Top mESC

,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,NEUROG3 OE MOUSE GSE3653 CREEDSID GENE 1660 DOWN,-2.484484,0.0,0.0,0.0
1,NEUROG3 OE MOUSE GSE3653 CREEDSID GENE 2289 DOWN,-2.483431,0.0,0.0,0.0
2,MTF2 KD MOUSE GSE16364 CREEDSID GENE 1355 UP,-2.412573,0.0,0.0,0.0
3,ZFP42 KD MOUSE GSE9978 CREEDSID GENE 1334 UP,-2.369883,0.0,0.0,0.0
4,ZFP42 KD MOUSE GSE9978 CREEDSID GENE 1335 UP,-2.243264,0.0,0.0,0.0
5,ZFX KO MOUSE GSE7069 CREEDSID GENE 177 DOWN,-2.213218,0.0,0.0,0.0
6,MSGN1 OE MOUSE GSE29848 CREEDSID GENE 1629 DOWN,-2.197723,0.0,0.0,0.0
7,SRF MUT MOUSE GSE1948 CREEDSID GENE 1400 DOWN,-2.134755,0.0,0.0,0.0
8,SOX2 KD MOUSE GSE39771 CREEDSID GENE 1757 DOWN,-2.1341,0.0,0.0,0.0
9,FOXD3 OE MOUSE GSE58960 CREEDSID GENE 1415 DOWN,-2.126921,0.0,0.0,0.0


## Test if specific gene sets are enriched

In [49]:
# Get libraries

panglao_lib = gseapy.get_library(name='PanglaoDB_Augmented_2021', organism='Mouse')
go_lib = gseapy.get_library(name='GO_Biological_Process_2025', organism='Mouse')
go_molecular_function = gseapy.get_library(name='GO_Molecular_Function_2025', organism='Mouse')
hallmark_lib = gseapy.get_library(name='MSigDB_Hallmark_2020', organism='Mouse')

In [51]:
target_sets = {
    "Apoptosis Activation": go_lib["Positive Regulation of Apoptotic Process (GO:0043065)"],
    "Apoptosis Inhibition": go_lib["Negative Regulation of Apoptotic Process (GO:0043066)"],
    "Cell Cycle Inhibition": go_lib["Negative Regulation of Cell Cycle (GO:0045786)"],
    "Cell Cycle Activation": go_lib["Positive Regulation of Cell Cycle (GO:0045787)"],
    "Apoptosis Signaling Inhibition": go_lib["Negative Regulation of Apoptotic Signaling Pathway (GO:2001234)"],
    "Apoptosis Signaling Activation": go_lib["Positive Regulation of Apoptotic Signaling Pathway (GO:2001235)"],
    "Stem Cell Proliferation": go_lib["Positive Regulation of Stem Cell Proliferation (GO:2000648)"],
    "Stem Cell Inhibited Proliferation": go_lib["Negative Regulation of Stem Cell Population Maintenance (GO:1902455)"],
    "Stem Cell Inhibited Differentiation": go_lib["Negative Regulation of Stem Cell Differentiation (GO:2000737)"],
    "Stem Cell Differentiation": go_lib["Positive Regulation of Stem Cell Differentiation (GO:2000738)"],
    "DNA Damage Response": go_lib["DNA Damage Response (GO:0006974)"],
    "DNA_Repair": hallmark_lib['DNA Repair'],
    "P53_Pathway": hallmark_lib['p53 Pathway'],
    "Keratinocytes": panglao_lib["Keratinocytes"],
    "Epithelial cells": panglao_lib["Epithelial Cells"]
}

rank_data = driver_df[["Day 12 Control_corr"]].sort_values('Day 12 Control_corr', ascending=False).dropna()

# Run GSEA Prerank
pre_res = gseapy.prerank(
    rnk=rank_data, 
    gene_sets=target_sets,
    threads=16,
    min_size=5,
    max_size=2000,
    permutation_num=1000,
    seed=0
)

pre_res.res2d.sort_values("NES", ascending=False).head(20)

2026-03-06 08:15:33,259 [WARNING] Duplicated values found in preranked stats: 0.15% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
0,prerank,Keratinocytes,0.624423,2.04658,0.0,0.0,0.0,21/41,15.77%,Lgals7;Lad1;Pkp1;Perp;Ovol1;Anxa8;S100a14;Krt1...
1,prerank,Epithelial cells,0.541963,1.944477,0.0,0.000591,0.001,29/67,16.78%,Lad1;Crb3;Mal2;Ovol1;Krt8;Dgat2;Gpx2;Misp;S100...
2,prerank,P53_Pathway,0.465663,1.624594,0.005714,0.033498,0.079,24/60,18.58%,Ccnd2;Ifi30;Nupr1;Perp;Cdkn1a;Sat1;Gpx2;Krt17;...
3,prerank,Apoptosis Signaling Activation,0.655181,1.540787,0.048951,0.045813,0.14,5/11,16.68%,Nupr1;Bbc3;App;Tnfrsf12a;Ddit3
7,prerank,Apoptosis Inhibition,0.210176,0.803516,0.957187,0.966857,0.98,18/107,15.02%,Ccnd2;Nupr1;Prkaa2;Cd44;Myc;Anxa5;Tmbim1;Ddah2...
10,prerank,Cell Cycle Inhibition,0.278801,0.698768,0.899314,0.92197,0.989,1/14,1.45%,Nupr1
12,prerank,Stem Cell Proliferation,-0.276325,-0.560267,0.96587,0.983849,1.0,1/8,7.81%,Nanog
11,prerank,Apoptosis Signaling Inhibition,-0.250039,-0.584337,0.988411,1.0,1.0,4/14,32.65%,Tcf7l2;Bmp4;Thbs1;Clu
9,prerank,Cell Cycle Activation,-0.292296,-0.706391,0.9018,1.0,0.999,2/16,4.81%,Tcf7l1;Cited2
8,prerank,Apoptosis Activation,-0.246637,-0.797562,0.867717,1.0,0.999,28/63,36.55%,Dlc1;Rest;Gadd45b;Top2a;Bnip3;Sfrp1;Tsc22d1;Ec...


# Differential expression analysis with Scanpy

Scanpy uses a Wilcoxon rank sum test.

In [9]:
# Assign each cell to the most likely macrostate

macrostates = adata.uns["coarse_fwd"].obs["coarse_init_dist"].index.to_list()
macrostate_assignment = [macrostates[numpy.argmax(macrostate_probabilities)] for macrostate_probabilities in adata.obsm["macrostates_fwd_memberships"]]
adata.obs["macrostate"] = macrostate_assignment
adata.uns["macrostate_colors"] = [
    adata.uns["macrostates_fwd_colors"][5],
    adata.uns["macrostates_fwd_colors"][2],
    adata.uns["macrostates_fwd_colors"][3],
    adata.uns["macrostates_fwd_colors"][4],
    adata.uns["macrostates_fwd_colors"][1],
    adata.uns["macrostates_fwd_colors"][0],
    adata.uns["macrostates_fwd_colors"][6]
]

## Comparison between all macrostates

In [25]:
# perform differential expression using the standard wilcoxon rank-sum test
scanpy.tl.rank_genes_groups(
    adata,
    groupby="macrostate",
    method="wilcoxon",
    use_raw=False,
    key_added="macrostate_rank_genes"
)

# extract the results into a dataframe for a specific target macrostate
dead_end_rank_df = scanpy.get.rank_genes_groups_df(
    adata,
    key="macrostate_rank_genes",
    group="Day 12 Control"
)
dead_end_rank_df.index = dead_end_rank_df["names"]
dead_end_rank_df = dead_end_rank_df.drop(columns="names")

control_rank_df = scanpy.get.rank_genes_groups_df(
    adata,
    key="macrostate_rank_genes",
    group="Day 4 Control"
)
control_rank_df.index = control_rank_df["names"]
control_rank_df = control_rank_df.drop(columns="names")

In [26]:
dead_end_rank_df.head(20)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Zfas1,26.189415,2.912675,3.507877e-151,7.015755e-148
2410006H16Rik,25.884188,2.383039,1.003519e-147,1.003519e-144
Snhg1,24.932861,2.789920,3.276535e-137,2.184357e-134
1110038B12Rik,24.600237,2.454037,1.255827e-133,6.279133e-131
Snhg12,24.165182,3.138035,5.170944e-129,2.068378e-126
Gas5,23.397280,1.672911,4.554957e-121,1.518319e-118
Ifrd1,22.545397,2.568011,1.490053e-112,4.257295e-110
Cdkn1a,21.841169,2.205297,9.431106e-106,2.357776e-103
Ppp1r15a,21.520761,2.846743,9.951511e-103,2.211447e-100


In [55]:
# Most upregulated transcription factors in dead end

dead_end_rank_df.loc[numpy.intersect1d(tf_array, adata.var_names)].sort_values("scores", ascending=False).head(30)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Atf3,18.789471,4.924244,9.209480e-79,8.372254e-77
Maff,14.781541,3.392600,1.927055e-49,5.416949e-48
Ddit3,13.226619,2.166627,6.159631e-40,1.140672e-38
Myc,12.562806,1.828156,3.381229e-36,5.162181e-35
Ovol1,12.100735,2.972715,1.046704e-33,1.443730e-32
Klf10,12.000822,1.918085,3.517865e-33,4.753872e-32
Arid3a,10.603693,1.693256,2.864403e-26,2.822072e-25
Zbtb7c,10.598906,1.849738,3.014877e-26,2.955762e-25
Junb,10.341942,1.311702,4.552173e-25,4.234580e-24


Really exciting! The top three TFs are involved in stress response, suggesting that this macrostate corresponds to high activity of 

In [56]:
hic2_target_scores.loc["Klf10"]

chip_score    4.637256
Name: Klf10, dtype: float64

In [57]:
# Most upregulated transcription factors in dead end that are shared targets

dead_end_rank_df.loc[numpy.intersect1d(target_tfs, adata.var_names)].sort_values("scores", ascending=False).head(30)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Ddit3,13.226619,2.166627,6.159631e-40,1.140672e-38
Myc,12.562806,1.828156,3.381229e-36,5.162181e-35
Ovol1,12.100735,2.972715,1.046704e-33,1.443730e-32
Klf10,12.000822,1.918085,3.517865e-33,4.753872e-32
Arid3a,10.603693,1.693256,2.864403e-26,2.822072e-25
Zbtb7c,10.598906,1.849738,3.014877e-26,2.955762e-25
Junb,10.341942,1.311702,4.552173e-25,4.234580e-24
Klf4,8.294026,1.242020,1.094780e-16,6.185198e-16
Phox2a,7.614943,2.153333,2.638079e-14,1.253244e-13


In [58]:
# Most downregulated transcription factors in dead end

dead_end_rank_df.loc[numpy.intersect1d(tf_array, adata.var_names)].sort_values("scores", ascending=True).head(30)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Cenpa,-14.694151,-2.029778,7.027420e-49,1.873979e-47
Hmgb2,-12.436042,-1.655495,1.665552e-35,2.485899e-34
Tcf7l1,-10.125731,-1.699106,4.247917e-24,3.646281e-23
Prrx1,-9.667671,-3.502777,4.136854e-22,3.182196e-21
Jarid2,-9.106974,-1.539088,8.471185e-20,5.782379e-19
Terf1,-8.995116,-1.125859,2.359814e-19,1.578471e-18
Nfib,-8.488363,-2.268551,2.095683e-17,1.240049e-16
Prrx2,-8.243245,-5.525078,1.676023e-16,9.337178e-16
Zeb1,-8.197787,-3.580868,2.448527e-16,1.345345e-15


In [59]:
# Most upregulated transcription factors in day 4 control

control_rank_df.loc[numpy.intersect1d(tf_array, adata.var_names)].sort_values("scores", ascending=False).head(30)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Tlx2,56.619308,4.156113,0.000000e+00,0.000000e+00
Dmrtc2,47.219395,3.756907,0.000000e+00,0.000000e+00
Zbtb7c,43.116035,2.244280,0.000000e+00,0.000000e+00
Hmgb3,39.926929,1.578542,0.000000e+00,0.000000e+00
Myc,39.168621,1.562202,0.000000e+00,0.000000e+00
Ovol1,35.878239,3.140644,6.674091e-282,1.026783e-280
Zbtb32,35.045231,2.059280,4.609327e-269,6.778422e-268
Hoxa9,28.716011,1.644503,2.407959e-181,2.239962e-180
Phox2a,28.670206,2.174598,8.977100e-181,8.273825e-180


In [13]:
dead_end_rank_df.loc[genes_of_interest].sort_values("scores", ascending=False)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Cdh1,13.483566,1.765469,1.954279e-41,3.908557e-40
Myc,12.562806,1.828156,3.381229e-36,5.162181e-35
Ovol1,12.100735,2.972715,1.046704e-33,1.443730e-32
Zbtb7c,10.598906,1.849738,3.014877e-26,2.955762e-25
Klf4,8.294026,1.242020,1.094780e-16,6.185198e-16
Tlx2,4.417089,0.874272,1.000390e-05,2.513543e-05
Hic2,-1.219937,-0.087063,2.224888e-01,3.171616e-01
Rest,-5.134597,-0.644595,2.827485e-07,8.124958e-07
Zfp42,-7.309782,-1.655304,2.675775e-13,1.197215e-12


## GSEA for dead end

In [61]:
# Cell types in the dead end

pre_res = gseapy.prerank(
    rnk=dead_end_rank_df[["scores"]], 
    gene_sets="PanglaoDB_Augmented_2021",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-06 08:16:03,198 [WARNING] Duplicated values found in preranked stats: 3.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,Mammary Epithelial Cells,2.588875,0.0,0.0,0.0
1,Trichocytes,2.454958,0.0,0.0,0.0
2,Keratinocytes,2.450678,0.0,0.0,0.0
3,Luminal Epithelial Cells,2.435519,0.0,0.0,0.0
4,Gastric Chief Cells,2.295272,0.0,0.0,0.0
5,Principal Cells,2.268925,0.0,0.0,0.0
6,Cholangiocytes,2.253806,0.0,0.0,0.0
7,Salivary Mucous Cells,2.175057,0.0,0.0,0.0
8,Sebocytes,2.173433,0.0,0.0,0.0
9,Airway Goblet Cells,2.135118,0.0,0.0,0.0


In [62]:
# MSig Hallmarks in dead end

pre_res = gseapy.prerank(
    rnk=dead_end_rank_df[["scores"]], 
    gene_sets="MSigDB_Hallmark_2020",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-06 08:16:10,767 [WARNING] Duplicated values found in preranked stats: 3.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,p53 Pathway,2.27238,0.0,0.0,0.0
3,TNF-alpha Signaling via NF-kB,1.981754,0.0,0.003,0.002193
6,Estrogen Response Early,1.623119,0.0,0.2,0.107212
7,UV Response Up,1.613178,0.005952,0.213,0.087354
8,mTORC1 Signaling,1.591396,0.006135,0.24,0.082164
9,Hypoxia,1.571308,0.0,0.268,0.079678
12,Androgen Response,1.432537,0.041056,0.565,0.18066
14,KRAS Signaling Dn,1.3821,0.082621,0.671,0.221126
17,Estrogen Response Late,1.338374,0.04893,0.782,0.254548
18,Unfolded Protein Response,1.335315,0.135135,0.788,0.232895


In [ ]:
# GO biological process in dead end

pre_res = gseapy.prerank(
    rnk=dead_end_rank_df[["scores"]], 
    gene_sets="GO_Biological_Process_2025",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-06 08:16:11,517 [WARNING] Duplicated values found in preranked stats: 3.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,Positive Regulation of Intrinsic Apoptotic Sig...,1.760335,0.007937,0.85,1.0
5,Regulation of Autophagy (GO:0010506),1.720115,0.004975,0.953,1.0
10,Antigen Processing and Presentation of Peptide...,1.702887,0.007895,0.971,0.841609
11,Antigen Processing and Presentation of Exogeno...,1.702887,0.007895,0.971,0.841609
9,Antigen Processing and Presentation of Exogeno...,1.702887,0.007895,0.971,0.841609
15,Positive Regulation of Sprouting Angiogenesis ...,1.646528,0.023018,0.996,1.0
18,Intracellular Iron Ion Homeostasis (GO:0006879),1.634716,0.026247,0.999,1.0
19,Regulation of Glycolytic Process (GO:0006110),1.62813,0.028796,0.999,1.0
20,Antigen Processing and Presentation of Peptide...,1.623347,0.018182,0.999,1.0
24,Positive Regulation of Neuron Apoptotic Proces...,1.609309,0.039578,1.0,1.0


In [ ]:
# TFs whose targets are upregulated in dead end, perturbation followed by expression
pre_res = gseapy.prerank(
    rnk=dead_end_rank_df[["scores"]], 
    gene_sets="TF_Perturbations_Followed_by_Expression",
    threads=32,
    min_size=5,
    max_size=1000
)
print("Top")
IPython.display.display(pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]])
print("Bottom")
IPython.display.display(pre_res.res2d.sort_values("NES", ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]])

2026-03-06 08:53:52,343 [WARNING] Duplicated values found in preranked stats: 3.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


Top


,Term,NES,NOM p-val,FWER p-val,FDR q-val
108,MYC OE MOUSE GSE55272 CREEDSID GENE 1822 DOWN,1.967528,0.0,0.056,0.042759
142,IRF6 KO MOUSE GSE5800 CREEDSID GENE 153 UP,1.914332,0.0,0.12,0.045275
143,PLAGL2 DEFICIENCY MOUSE GSE9123 CREEDSID GENE ...,1.913938,0.0,0.12,0.030393
154,ARID3A KD MOUSE GSE56853 CREEDSID GENE 1337 UP,1.90636,0.0,0.135,0.025467
159,HEY2 KO MOUSE GSE6526 CREEDSID GENE 1511 UP,1.896851,0.0,0.156,0.019598
158,HEY2 KO MOUSE GSE6526 CREEDSID GENE 1512 DOWN,1.896851,0.0,0.156,0.019598
171,PPARA KO MOUSE GSE6864 CREEDSID GENE 981 UP,1.874495,0.0,0.217,0.023626
176,FOXO1 KD MOUSE GSE6623 CREEDSID GENE 505 DOWN,1.870261,0.0,0.23,0.022166
187,GLI1 INHIBITION HUMAN GSE36855 CREEDSID GENE 6...,1.858362,0.0,0.257,0.022498
200,PPARA KO MOUSE GSE6864 CREEDSID GENE 1371 DOWN,1.840598,0.0,0.308,0.025027


Bottom


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,FOXA1 KD HUMAN GSE25315 CREEDSID GENE 2164 DOWN,-2.463419,0.0,0.0,0.0
1,OVOL2 OE PC3 HUMAN GSE48230 RNASEQ DOWN,-2.452743,0.0,0.0,0.0
2,DNMT1 KO MOUSE GSE31626 CREEDSID GENE 1237 DOWN,-2.417583,0.0,0.0,0.0
3,E2F1 OE HUMAN GSE2715 CREEDSID GENE 1129 DOWN,-2.407181,0.0,0.0,0.0
4,TAL1 KD JURKAT HUMAN GSE72299 RNASEQ DOWN,-2.372448,0.0,0.0,0.0
5,MYB KD HUMAN GSE49286 CREEDSID GENE 1842 DOWN,-2.353719,0.0,0.0,0.0
6,MEIS2 KD KASUMI1 HUMAN GSE81328 RNASEQ UP,-2.337821,0.0,0.0,0.0
7,ZNF750 KD HUMAN GSE38039 CREEDSID GENE 204 UP,-2.336913,0.0,0.0,0.0
8,ZNF750 KD HUMAN GSE38039 CREEDSID GENE 2704 DOWN,-2.336913,0.0,0.0,0.0
9,ZMAT4 SIRNA T47D HUMAN GSE79586 RNASEQ DOWN,-2.332155,0.0,0.0,0.0


In [146]:
# TFs whose targets are upregulated in dead end, Chea
pre_res = gseapy.prerank(
    rnk=dead_end_rank_df[["scores"]], 
    gene_sets="ChEA_2022",
    threads=32,
    min_size=5,
    max_size=1000
)
print("Top")
IPython.display.display(pre_res.res2d.sort_values("NES", ascending=False).head(30)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]])
print("Bottom")
IPython.display.display(pre_res.res2d.sort_values("NES", ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]])

2026-03-06 09:15:54,894 [WARNING] Duplicated values found in preranked stats: 3.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


Top


,Term,NES,NOM p-val,FWER p-val,FDR q-val
3,TP63 17297297 ChIP-ChIP HaCaT Human,1.829356,0.008772,0.081,0.084076
8,FOXA1 33576154 ChIP-Seq Human HilarLN Prostate...,1.684291,0.003472,0.298,0.182961
14,ESR1 26153859 ChIP-Seq MCF-7 Human BreastCancer,1.611439,0.0,0.513,0.245222
17,NOTCH1 21737748 ChIP-Seq TLL Human,1.595993,0.012195,0.573,0.214012
22,RELB 30642670 ChIP-Seq CTB1 Human Placenta Inf...,1.566168,0.008902,0.661,0.221655
25,HCFC1 20581084 ChIP-Seq MESCs Mouse,1.55914,0.046753,0.684,0.196655
31,NUCKS1 24931609 ChIP-Seq HEPATOCYTES Mouse,1.539307,0.0,0.742,0.200363
36,CDX2 20551321 ChIP-Seq CACO-2 Human,1.516565,0.006173,0.815,0.213415
37,ELF1 20517297 ChIP-Seq JURKAT Human,1.510851,0.0,0.83,0.200212
38,TP63 30713093 ChIP-Seq Epithelial Human Tongue...,1.503021,0.0,0.846,0.192802


Bottom


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,FOXM1 23109430 ChIP-Seq U2OS Human,-2.304463,0.0,0.0,0.0
1,FOXM1 25889361 ChIP-Seq OE33 AND U2OS Human,-2.008273,0.0,0.001,0.000508
2,E2F4 17652178 ChIP-ChIP JURKAT Human,-1.948152,0.0,0.002,0.000678
4,EKLF 21900194 ChIP-Seq ERYTHROCYTE Mouse,-1.779874,0.0,0.064,0.018295
5,E2F4 21247883 ChIP-Seq LYMPHOBLASTOID Human,-1.766155,0.0,0.077,0.017482
6,FOXM1 26456572 ChIP-Seq MCF-7 Human BreastCancer,-1.76615,0.0,0.077,0.014569
7,GABP 19822575 ChIP-Seq HepG2 Human,-1.735868,0.0,0.132,0.022071
9,RUNX2 24655370 ChIP-Seq MC3T3E1 Mouse Bone,-1.662657,0.0,0.326,0.054632
10,MYBL2 22936984 ChIP-ChIP MESCs Mouse,-1.65339,0.0,0.366,0.056467
11,EWS 26573619 Chip-Seq HEK293 Human,-1.631371,0.0,0.456,0.069726


In [147]:
# TFs whose targets are upregulated in dead end, ENCODE
pre_res = gseapy.prerank(
    rnk=dead_end_rank_df[["scores"]], 
    gene_sets="ENCODE_TF_ChIP-seq_2015",
    threads=32,
    min_size=5,
    max_size=1000
)
print("Top")
IPython.display.display(pre_res.res2d.sort_values("NES", ascending=False).head(30)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]])
print("Bottom")
IPython.display.display(pre_res.res2d.sort_values("NES", ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]])

2026-03-06 09:16:10,062 [WARNING] Duplicated values found in preranked stats: 3.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


Top


,Term,NES,NOM p-val,FWER p-val,FDR q-val
4,MAX myocyte mm9,1.941958,0.0,0.027,0.028555
7,NR2F2 MCF-7 hg19,1.860825,0.0,0.062,0.034164
10,ZEB1 HepG2 hg19,1.805283,0.0,0.112,0.043173
14,FOSL2 HepG2 hg19,1.788335,0.0,0.129,0.037734
16,YY1 GM12891 hg19,1.748837,0.0,0.19,0.045893
20,ATF3 K562 hg19,1.711503,0.0,0.265,0.057111
22,PML MCF-7 hg19,1.703937,0.00304,0.281,0.05274
25,TAF1 MCF-7 hg19,1.677512,0.0,0.345,0.060425
27,JUND MCF-7 hg19,1.671517,0.005556,0.359,0.056431
29,REST HeLa-S3 hg19,1.664067,0.0,0.376,0.054153


Bottom


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,FOXM1 ECC-1 hg19,-2.328247,0.0,0.0,0.0
1,E2F4 MEL cell line mm9,-2.104461,0.0,0.0,0.0
2,E2F4 HeLa-S3 hg19,-2.030328,0.0,0.0,0.0
3,FOXM1 MCF-7 hg19,-1.976726,0.0,0.002,0.000496
5,IRF3 HeLa-S3 hg19,-1.907565,0.0,0.007,0.001587
6,NFYA GM12878 hg19,-1.879223,0.0,0.012,0.002149
8,SP2 HepG2 hg19,-1.841056,0.0,0.027,0.003968
9,SP2 H1-hESC hg19,-1.831186,0.0,0.034,0.004463
11,IRF3 GM12878 hg19,-1.800409,0.0,0.055,0.006613
12,E2F4 CH12.LX mm9,-1.797015,0.0,0.058,0.006249


In [148]:
# Motif enrichment analysis, TRANSFAC and JASPAR

# Motifs upregulated in dead end
pre_res = gseapy.prerank(
    rnk=dead_end_rank_df[["scores"]], 
    gene_sets="TRANSFAC_and_JASPAR_PWMs",
    threads=32,
    min_size=5,
    max_size=1000
)
print("Top")
IPython.display.display(pre_res.res2d.sort_values("NES", ascending=False).head(30)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]])
print("Bottom")
IPython.display.display(pre_res.res2d.sort_values("NES", ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]])

2026-03-06 09:16:21,924 [WARNING] Duplicated values found in preranked stats: 3.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


Top


,Term,NES,NOM p-val,FWER p-val,FDR q-val
2,SNAI1 (human),1.698159,0.0,0.217,0.125241
1,TCF3 (human),1.698159,0.0,0.217,0.125241
3,SNAI2 (human),1.686804,0.0,0.24,0.095421
9,CEBPB (human),1.45279,0.0,0.812,0.4453
12,NCOA1 (human),1.438102,0.094737,0.84,0.398782
13,MIR210 (human),1.431728,0.077135,0.854,0.347228
14,TFAP2C (human),1.431336,0.008097,0.856,0.299328
24,MYOD1 (human),1.334539,0.126344,0.967,0.491271
25,MIB2 (human),1.328755,0.015385,0.974,0.454577
28,SMAD4 (human),1.306174,0.008163,0.987,0.47661


Bottom


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,Nkx2-5 (mouse),-1.743336,0.00321,0.046,0.047112
4,HNF1B (human),-1.657876,0.004808,0.191,0.10475
5,SOX10 (human),-1.498589,0.022117,0.783,0.528927
6,NFYA (mouse),-1.479768,0.002688,0.832,0.485407
7,NRF1 (human),-1.456635,0.008559,0.892,0.498789
8,HIVEP1 (human),-1.456613,0.025994,0.892,0.415991
10,EIF4EBP1 (human),-1.446871,0.05414,0.919,0.3958
11,SP3 (mouse),-1.439081,0.002685,0.932,0.377399
15,SRF (mouse),-1.412439,0.020891,0.971,0.444504
16,Nr2e3 (mouse),-1.40943,0.059295,0.971,0.411781


In [149]:
# Motifs upregulated in dead end, 
pre_res = gseapy.prerank(
    rnk=dead_end_rank_df[["scores"]], 
    gene_sets="JASPAR_PWM_Mouse_2025",
    threads=32,
    min_size=5,
    max_size=1000
)
print("Top")
IPython.display.display(pre_res.res2d.sort_values("NES", ascending=False).head(30)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]])
print("Bottom")
IPython.display.display(pre_res.res2d.sort_values("NES", ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]])

2026-03-06 09:16:34,766 [WARNING] Duplicated values found in preranked stats: 3.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


Top


,Term,NES,NOM p-val,FWER p-val,FDR q-val
8,TCF12,1.350296,0.0,0.345,0.349353
9,NPAS2,1.341644,0.037162,0.369,0.190975
25,HAND1,1.237528,0.032864,0.671,0.378261
31,RARB,1.20734,0.052632,0.754,0.387486
44,CREB3L2,1.146858,0.140893,0.892,0.552199
49,NR1H4,1.097714,0.174468,0.955,0.716235
59,TFCP2L1,1.06455,0.239521,0.98,0.834458
64,ARNTL,1.038151,0.385093,0.993,0.903827
73,AHR::ARNT,1.007965,0.374016,0.998,1.0
74,SPZ1,1.005296,0.402597,0.998,0.931443


Bottom


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,ARX,-1.623406,0.0,0.051,0.065311
1,SOX11,-1.596687,0.0,0.072,0.046731
2,ALX1,-1.563339,0.002618,0.104,0.046919
3,ALX4,-1.441919,0.015603,0.385,0.172568
4,LHX4,-1.410246,0.026201,0.493,0.213049
5,STAT2,-1.407999,0.007732,0.502,0.182421
6,GFI1B,-1.407851,0.001261,0.502,0.156682
7,FOXF1,-1.37432,0.009695,0.654,0.220847
10,SOX17,-1.329395,0.01715,0.789,0.343822
11,SOX5,-1.325497,0.031208,0.802,0.322614


## GSEA for day 4 control

In [135]:
# Cell types in control

pre_res = gseapy.prerank(
    rnk=control_rank_df[["scores"]], 
    gene_sets="PanglaoDB_Augmented_2021",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-06 08:57:38,496 [WARNING] Duplicated values found in preranked stats: 0.55% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,Epithelial Cells,2.172779,0.0,0.0,0.0
1,Keratinocytes,2.171084,0.0,0.0,0.0
2,Mammary Epithelial Cells,2.167204,0.0,0.0,0.0
3,Gastric Chief Cells,2.129438,0.0,0.0,0.0
4,Luminal Epithelial Cells,2.125696,0.0,0.0,0.0
5,Cholangiocytes,2.077497,0.0,0.0,0.0
6,Trichocytes,2.049509,0.0,0.001,0.000147
9,Hepatoblasts,1.97248,0.001916,0.003,0.000387
13,Principal Cells,1.939494,0.0,0.006,0.000688
14,Salivary Mucous Cells,1.938638,0.0,0.006,0.000619


In [63]:
# MSig Hallmarks in day 4 control

pre_res = gseapy.prerank(
    rnk=control_rank_df[["scores"]], 
    gene_sets="MSigDB_Hallmark_2020",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-06 08:16:11,166 [WARNING] Duplicated values found in preranked stats: 0.55% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,Estrogen Response Late,1.650953,0.001751,0.15,0.162436
2,Estrogen Response Early,1.568822,0.010363,0.318,0.192184
4,Myc Targets V1,1.406796,0.09589,0.753,0.486363
5,E2F Targets,1.397855,0.062278,0.771,0.390271
7,p53 Pathway,1.348818,0.069027,0.861,0.429322
8,Wnt-beta Catenin Signaling,1.346102,0.142562,0.865,0.364536
9,mTORC1 Signaling,1.345051,0.075134,0.865,0.314888
10,Notch Signaling,1.342163,0.138493,0.87,0.278478
11,Xenobiotic Metabolism,1.282317,0.109319,0.947,0.353098
12,Androgen Response,1.269339,0.15283,0.958,0.341587


In [134]:
# TFs whose targets are affected in control
pre_res = gseapy.prerank(
    rnk=control_rank_df[["scores"]], 
    gene_sets="TF_Perturbations_Followed_by_Expression",
    threads=32,
    min_size=5,
    max_size=1000
)
print("Top")
IPython.display.display(pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]])
print("Bottom")
IPython.display.display(pre_res.res2d.sort_values("NES", ascending=True).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]])

2026-03-06 08:54:49,726 [WARNING] Duplicated values found in preranked stats: 0.55% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


Top


,Term,NES,NOM p-val,FWER p-val,FDR q-val
7,OVOL2 OE MOUSE GSE55074 CREEDSID GENE 2603 DOWN,2.05011,0.0,0.005,0.003324
9,MYC OE MOUSE GSE55272 CREEDSID GENE 1822 DOWN,2.039413,0.0,0.007,0.002327
10,IRF6 KO MOUSE GSE5800 CREEDSID GENE 153 UP,2.039194,0.0,0.007,0.001551
38,HSF1 KO MOUSE GSE41005 CREEDSID GENE 2143 DOWN,1.856263,0.0,0.234,0.048202
39,YY1 KD MOUSE GSE31784 CREEDSID GENE 1333 UP,1.850319,0.0,0.257,0.042418
56,MBD3 KD MOUSE GSE31008 CREEDSID GENE 1325 UP,1.805803,0.001669,0.429,0.070475
62,PPARA DEFICIENCY MOUSE GSE6864 CREEDSID GENE 6...,1.793981,0.0,0.474,0.06943
63,KLF1 KO MOUSE GSE36427 CREEDSID GENE 1562 DOWN,1.790153,0.001855,0.496,0.06499
65,NFE2L2 KO MOUSE GSE18344 CREEDSID GENE 965 DOWN,1.785133,0.0,0.512,0.061093
78,HNF4A MUT MOUSE GSE3116 CREEDSID GENE 850 DOWN,1.763923,0.0,0.604,0.073135


Bottom


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,TCF3 KO MOUSE GSE19923 CREEDSID GENE 2167 UP,-2.135971,0.0,0.003,0.004398
1,MYC KD HUMAN GSE22139 CREEDSID GENE 712 UP,-2.114692,0.0,0.003,0.002199
2,NEUROG3 OE MOUSE GSE3653 CREEDSID GENE 2289 DOWN,-2.108357,0.0,0.003,0.001466
3,ZFX KO MOUSE GSE7069 CREEDSID GENE 177 UP,-2.102751,0.0,0.004,0.001466
4,MYC OE HUMAN GSE43730 CREEDSID GENE 359 DOWN,-2.068408,0.0,0.004,0.001173
5,NEUROG3 OE MOUSE GSE3653 CREEDSID GENE 1660 DOWN,-2.064367,0.0,0.004,0.000977
6,CDX2 KO MOUSE GSE12999 CREEDSID GENE 773 UP,-2.056295,0.0,0.004,0.000838
8,EBF1 OE MOUSE GSE2192 CREEDSID GENE 859 DOWN,-2.046406,0.0,0.004,0.000733
11,GATA3 OE MOUSE GSE12999 CREEDSID GENE 2531 DOWN,-2.031703,0.0,0.007,0.001303
12,REST SHRNA C2 HUMAN GSE90068 PBPA RNASEQ DOWN,-2.016792,0.0,0.014,0.002199


These results suggest that the control pathway has low Ovol2 activity, which is consistent with high Ovol1 activity as Ovol1 downregulates Ovol2.

## Compare day 12 control with day 4 control

In [68]:
# Compare day 12 control with day 4 control
scanpy.tl.rank_genes_groups(
    adata,
    groupby="macrostate",
    method="wilcoxon",
    use_raw=False,
    reference="Day 4 Control",
    key_added="macrostate_vs_control_rank_genes"
)
dead_end_vs_control_rank_df = scanpy.get.rank_genes_groups_df(
    adata,
    group="Day 12 Control",
    key="macrostate_vs_control_rank_genes"
)
dead_end_vs_control_rank_df.index = dead_end_vs_control_rank_df["names"]
dead_end_vs_control_rank_df = dead_end_vs_control_rank_df.drop(columns="names")
dead_end_vs_control_rank_df.head(30)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
2410006H16Rik,26.126083,2.549994,1.843133e-150,3.686265e-147
Zfas1,25.591373,2.887306,1.903077e-144,1.903077e-141
1110038B12Rik,25.104904,2.706954,4.396385e-139,2.930924e-136
Snhg1,24.437115,2.785392,6.898948e-132,3.449474e-129
Gas5,24.352331,1.764712,5.476893e-131,2.190757e-128
Snhg12,24.150040,3.244454,7.459324e-129,2.486441e-126
Ppp1r15a,21.102528,2.884851,7.539659e-99,1.507932e-96
Sqstm1,19.848444,2.438090,1.136654e-87,1.623791e-85
Cstb,19.777744,1.662232,4.629430e-87,6.172574e-85


In [69]:
# Most upregulated transcription factors in dead end

dead_end_vs_control_rank_df.loc[numpy.intersect1d(tf_array, adata.var_names)].sort_values("scores", ascending=False).head(30)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Atf3,17.293951,4.182650,5.224525e-67,3.483017e-65
Ddit3,15.975071,3.221851,1.906363e-57,9.776220e-56
Maff,14.538376,3.518187,6.921518e-48,2.307173e-46
Crebrf,11.278059,3.426975,1.684214e-29,2.716475e-28
Zbtb10,9.688887,2.093428,3.361729e-22,3.954976e-21
Junb,9.540514,1.198861,1.421259e-21,1.615067e-20
Arid3a,9.539727,1.519866,1.432088e-21,1.618178e-20
Klf6,9.515185,1.901875,1.813887e-21,2.026689e-20
Klf4,9.149061,1.463191,5.743040e-20,5.771900e-19


Top 4 TFs are all involved in stress responses

In [70]:
# Most downregulated transcription factors in dead end

dead_end_vs_control_rank_df.loc[numpy.intersect1d(tf_array, adata.var_names)].sort_values("scores", ascending=True).head(30)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Cenpa,-18.378662,-2.361419,1.947120e-75,1.557696e-73
Hmgb3,-14.654617,-2.041724,1.258640e-48,4.266575e-47
Hmgb2,-11.795198,-1.436325,4.132265e-32,7.313744e-31
Id2,-8.479613,-1.096260,2.259429e-17,1.990686e-16
Tlx2,-8.452542,-1.106642,2.850273e-17,2.500239e-16
Prrx1,-8.317663,-2.595609,8.971450e-17,7.700815e-16
Mis18bp1,-7.335813,-1.151761,2.203806e-13,1.541123e-12
Terf1,-7.233290,-0.849773,4.714325e-13,3.217969e-12
Zfhx4,-6.941576,-1.290629,3.877483e-12,2.493558e-11


In [71]:
dead_end_vs_control_rank_df.loc[genes_of_interest].sort_values("scores", ascending=False)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Klf4,9.149061,1.463191,5.743040e-20,5.771900e-19
Ovol1,6.053084,1.239698,1.420988e-09,7.722761e-09
Myc,5.996467,0.733084,2.016562e-09,1.084173e-08
Rest,3.244519,0.294803,1.176493e-03,3.435017e-03
Zbtb7c,3.167943,0.454898,1.535217e-03,4.405213e-03
Hic2,1.094972,0.462989,2.735291e-01,4.687730e-01
Tcf7l1,-4.097703,-0.527595,4.172700e-05,1.498276e-04
Zfp42,-4.107964,-0.588239,3.991626e-05,1.441020e-04
Jarid2,-4.346265,-0.551580,1.384752e-05,5.255225e-05


The dead end state is signified by high expression of Klf4, Myc and Ovol1.

Tlx2 is downregulated in the dead end, suggesting it does not bring cells into the dead end. It could just be correlated with the control pathway.

In [72]:
# Cell types in the dead end

pre_res = gseapy.prerank(
    rnk=dead_end_vs_control_rank_df[["scores"]],
    gene_sets="PanglaoDB_Augmented_2021",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-06 08:16:19,628 [WARNING] Duplicated values found in preranked stats: 9.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,Mammary Epithelial Cells,2.032868,0.0,0.0,0.0
1,Principal Cells,1.998692,0.0,0.0,0.0
2,Luminal Epithelial Cells,1.976177,0.0,0.003,0.001648
3,Trichocytes,1.862625,0.003597,0.028,0.010508
4,Gastric Chief Cells,1.796434,0.0,0.062,0.018296
6,Sebocytes,1.740809,0.009375,0.12,0.030699
8,Cholangiocytes,1.672694,0.006515,0.229,0.05563
9,Keratinocytes,1.635431,0.003597,0.303,0.071855
10,Airway Epithelial Cells,1.616102,0.02029,0.363,0.079118
11,Nuocytes,1.613408,0.035294,0.37,0.072689


In [73]:
# MSig Hallmarks

pre_res = gseapy.prerank(
    rnk=dead_end_vs_control_rank_df[["scores"]], 
    gene_sets="MSigDB_Hallmark_2020",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-06 08:16:19,865 [WARNING] Duplicated values found in preranked stats: 9.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,TNF-alpha Signaling via NF-kB,2.23291,0.0,0.0,0.0
1,p53 Pathway,2.080421,0.0,0.0,0.0
4,Hypoxia,1.837935,0.0,0.01,0.005158
7,UV Response Up,1.595374,0.0,0.175,0.075826
11,Unfolded Protein Response,1.436948,0.081301,0.477,0.21974
13,mTORC1 Signaling,1.398681,0.035484,0.576,0.239599
16,TGF-beta Signaling,1.300711,0.137931,0.792,0.37095
20,Estrogen Response Early,1.19468,0.152941,0.928,0.550124
21,Protein Secretion,1.17724,0.271186,0.936,0.531985
22,IL-6/JAK/STAT3 Signaling,1.161149,0.194529,0.945,0.512521


In [74]:
# GO biological process

pre_res = gseapy.prerank(
    rnk=dead_end_vs_control_rank_df[["scores"]], 
    gene_sets="GO_Biological_Process_2025",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-06 08:16:19,964 [WARNING] Duplicated values found in preranked stats: 9.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,Regulation of JNK Cascade (GO:0046328),1.805935,0.0,0.323,0.486163
1,Response to Endoplasmic Reticulum Stress (GO:0...,1.799198,0.003125,0.351,0.273211
2,Positive Regulation of JNK Cascade (GO:0046330),1.792325,0.0,0.382,0.202908
6,Macroautophagy (GO:0016236),1.670805,0.010929,0.913,0.949601
7,Anterograde Trans-Synaptic Signaling (GO:0098916),1.66677,0.012698,0.922,0.800535
8,Regulation of Translation (GO:0006417),1.651243,0.012012,0.959,0.820315
9,Positive Regulation of Granulocyte Chemotaxis ...,1.648447,0.010724,0.964,0.732454
10,Integrated Stress Response Signaling (GO:0140467),1.644288,0.002681,0.966,0.674985
13,Immunoglobulin Mediated Immune Response (GO:00...,1.632127,0.005602,0.977,0.697809
17,T Cell Chemotaxis (GO:0010818),1.615536,0.010724,0.984,0.766932


All the top terms are related to damage and stress signaling.

In [75]:
# ENCODE ChIP-Seq
pre_res = gseapy.prerank(
    rnk=dead_end_vs_control_rank_df[["scores"]], 
    gene_sets="ENCODE_TF_ChIP-seq_2015",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-06 08:16:21,105 [WARNING] Duplicated values found in preranked stats: 9.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
2,FOSL2 HepG2 hg19,1.845511,0.0,0.032,0.049575
5,MAX myocyte mm9,1.794046,0.0,0.061,0.048158
6,PML MCF-7 hg19,1.772045,0.003846,0.083,0.042493
7,NR2F2 MCF-7 hg19,1.756168,0.0,0.096,0.038952
10,TAF1 MCF-7 hg19,1.739986,0.0,0.117,0.039093
11,NR3C1 ECC-1 hg19,1.739029,0.003484,0.117,0.032814
14,NFE2 K562 hg19,1.696168,0.009009,0.193,0.046944
15,USF1 K562 hg19,1.691845,0.0,0.198,0.042316
16,YY1 GM12891 hg19,1.6876,0.007117,0.21,0.039975
17,MEF2A K562 hg19,1.660891,0.003185,0.265,0.047308


In [76]:
# Chsea
pre_res = gseapy.prerank(
    rnk=dead_end_vs_control_rank_df[["scores"]], 
    gene_sets="ChEA_2022",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-06 08:16:22,387 [WARNING] Duplicated values found in preranked stats: 9.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
2,FOXA1 33576154 ChIP-Seq Human HilarLN Prostate...,1.891787,0.0,0.013,0.01714
4,RELB 30642670 ChIP-Seq CTB1 Human Placenta Inf...,1.788053,0.003236,0.068,0.046145
6,TP63 17297297 ChIP-ChIP HaCaT Human,1.713096,0.003115,0.151,0.073832
8,NOTCH1 21737748 ChIP-Seq TLL Human,1.6068,0.018237,0.396,0.177659
9,PKCTHETA 26484144 Chip-Seq BREAST Human,1.601674,0.003676,0.414,0.150038
10,FOXA1 26457646 ChIP-Seq LHSAR Human ProstateCa...,1.591983,0.0,0.446,0.137337
11,FOXO1 32281255 ChIP-Seq Chondrocytes Human Ost...,1.590245,0.0,0.451,0.119036
12,MEIS1 20887958 ChIP-Seq HPC-7 Mouse,1.577726,0.0,0.489,0.116681
13,NUCKS1 24931609 ChIP-Seq HEPATOCYTES Mouse,1.572461,0.0,0.506,0.10899
14,BRD4 28847988 ChIP-Seq BCBL1 Human Blood Lymphoma,1.561826,0.019231,0.54,0.109166


In [77]:
# TF perturbation
pre_res = gseapy.prerank(
    rnk=dead_end_vs_control_rank_df[["scores"]], 
    gene_sets="TF_Perturbations_Followed_by_Expression",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-06 08:16:23,610 [WARNING] Duplicated values found in preranked stats: 9.00% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
2,GLI1 INHIBITION HUMAN GSE36855 CREEDSID GENE 6...,2.180205,0.0,0.0,0.0
20,FOXO1 KD MOUSE GSE6623 CREEDSID GENE 505 DOWN,1.988647,0.0,0.016,0.00618
29,SOX4 KD HUMAN GSE4225 CREEDSID GENE 64 UP,1.91645,0.0,0.061,0.016738
45,NFKB1 INACTIVATION HUMAN GSE20667 CREEDSID GEN...,1.857504,0.003115,0.132,0.02839
54,HEY2 KO MOUSE GSE6526 CREEDSID GENE 1511 UP,1.825931,0.0,0.204,0.031029
55,HEY2 KO MOUSE GSE6526 CREEDSID GENE 1512 DOWN,1.825931,0.0,0.204,0.031029
56,ZXDC DEPLET MOUSE GSE45417 CREEDSID GENE 1257 UP,1.819913,0.0,0.216,0.028693
62,PLAGL2 DEFICIENCY MOUSE GSE9123 CREEDSID GENE ...,1.806016,0.0,0.249,0.029838
79,CEBPA KO MOUSE GSE61468 CREEDSID GENE 1476 UP,1.775358,0.0,0.362,0.041458
82,ARID3B KO MOUSE GSE62069 CREEDSID GENE 2591 UP,1.772634,0.0,0.372,0.038703


## Other macrostates

### MEF

In [17]:
mef_rank_df = scanpy.get.rank_genes_groups_df(
    adata,
    key="macrostate_rank_genes",
    group="MEF"
)
mef_rank_df.index = mef_rank_df["names"]
mef_rank_df = mef_rank_df.drop(columns="names")

In [21]:
mef_rank_df.loc[numpy.intersect1d(target_tfs, adata.var_names)].sort_values("scores", ascending=False).head(20)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Tcf4,22.032331,3.015500,1.411167e-107,1.512375e-105
Prrx1,19.597130,3.227617,1.635962e-85,4.674177e-84
Snai2,18.127380,3.744907,1.937707e-73,3.429570e-72
Atf5,17.698309,2.213542,4.320908e-70,6.647551e-69
Creb3l1,17.047623,2.918983,3.640356e-65,4.919400e-64
Twist1,16.896849,2.630791,4.746084e-64,6.286205e-63
Ddit3,15.490962,2.365666,3.992722e-54,3.784570e-53
Nfix,14.685015,2.538224,8.041650e-49,6.185884e-48
Id3,13.723390,2.272552,7.354487e-43,4.699353e-42


In [22]:
mef_rank_df.loc[genes_of_interest].sort_values("scores", ascending=False).head(20)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Rest,0.831551,0.005710,4.056624e-01,5.154542e-01
Tcf7l1,-2.511146,-0.392221,1.203400e-02,1.904113e-02
Ovol1,-4.395990,-27.196196,1.102691e-05,2.184011e-05
Klf4,-4.772457,-1.034347,1.819917e-06,3.775761e-06
Myc,-5.386568,-1.016510,7.181592e-08,1.597685e-07
Tlx2,-6.352568,-7.615314,2.117498e-10,5.429482e-10
Zbtb7c,-7.334432,-3.338338,2.226635e-13,6.520161e-13
Hic2,-8.569461,-2.538984,1.039707e-17,3.494815e-17
Jarid2,-10.025214,-2.096941,1.181037e-23,4.752662e-23


In [49]:
# Cell types

pre_res = gseapy.prerank(
    rnk=mef_rank_df[["scores"]], 
    gene_sets="PanglaoDB_Augmented_2021",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)

2026-03-09 13:51:09,658 [WARNING] Duplicated values found in preranked stats: 2.20% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
4,prerank,Airway Smooth Muscle Cells,0.746741,2.232371,0.0,0.0,0.0,46/72,15.75%,Col12a1;Col5a2;Col3a1;Col5a1;Col1a1;Col1a2;Bgn...
5,prerank,Osteoblasts,0.712196,2.219959,0.0,0.0,0.0,73/100,26.35%,Col12a1;Col5a2;Col3a1;Col5a1;Col1a1;Col1a2;Bgn...
6,prerank,Myofibroblasts,0.738854,2.210997,0.0,0.0,0.0,46/73,15.65%,Col12a1;Col5a2;Col3a1;Col5a1;Col1a1;Col1a2;Bgn...
8,prerank,Pancreatic Stellate Cells,0.722599,2.185598,0.0,0.0,0.0,50/82,14.35%,Col12a1;Col5a2;Col3a1;Col5a1;Col1a1;Col1a2;Bgn...
10,prerank,Chondrocytes,0.703991,2.154509,0.0,0.0,0.0,53/94,16.50%,Col12a1;Col5a2;Col3a1;Col5a1;Col1a1;Col1a2;Bgn...
12,prerank,Hepatic Stellate Cells,0.701484,2.128608,0.0,0.0,0.0,47/84,15.65%,Col12a1;Col5a2;Col3a1;Col5a1;Col1a1;Col1a2;Bgn...
13,prerank,Mesangial Cells,0.686661,2.128004,0.0,0.0,0.0,60/100,21.40%,Col12a1;Col5a2;Col3a1;Col5a1;Col1a1;Col1a2;Bgn...
14,prerank,Myoblasts,0.710322,2.124545,0.0,0.0,0.0,37/61,18.70%,Col12a1;Col5a2;Col3a1;Col5a1;Col1a1;Col1a2;Bgn...
15,prerank,Vascular Smooth Muscle Cells,0.699649,2.097212,0.0,0.0,0.0,53/72,25.00%,Col12a1;Col5a2;Col3a1;Col5a1;Col1a1;Col1a2;Bgn...
16,prerank,Smooth Muscle Cells,0.681828,2.091595,0.0,0.0,0.0,51/92,19.40%,Col12a1;Col5a2;Col3a1;Col5a1;Col1a1;Col1a2;Bgn...


### Day 2 control

In [27]:
early_rank_df = scanpy.get.rank_genes_groups_df(
    adata,
    key="macrostate_rank_genes",
    group="Day 2 Control"
)
early_rank_df.index = early_rank_df["names"]
early_rank_df = early_rank_df.drop(columns="names")

In [34]:
early_rank_df.loc[genes_of_interest].sort_values("scores", ascending=False).head(20)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Tcf7l1,-5.049325,-0.307710,4.433749e-07,6.333927e-07
Rest,-9.274354,-0.426103,1.786991e-20,3.086340e-20
Klf4,-10.298006,-0.637572,7.193706e-25,1.296163e-24
Ovol1,-15.122104,-4.998144,1.157831e-51,2.570101e-51
Tlx2,-21.331772,-4.334157,5.758435e-101,1.683753e-100
Zbtb7c,-22.501848,-2.530346,3.981161e-112,1.230653e-111
Myc,-23.374834,-1.289301,7.706647e-121,2.506227e-120
Hic2,-25.126188,-1.832231,2.573806e-139,8.999320e-139
Cdh1,-38.846001,-4.595285,0.000000e+00,0.000000e+00


In [50]:
# Cell types

pre_res = gseapy.prerank(
    rnk=early_rank_df[["scores"]], 
    gene_sets="PanglaoDB_Augmented_2021",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)

2026-03-09 13:52:18,493 [WARNING] Duplicated values found in preranked stats: 0.10% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
5,prerank,Myoblasts,0.72998,2.200489,0.0,0.0,0.0,45/61,22.25%,Cald1;Col1a1;Col5a2;Col1a2;Tpm1;Fstl1;Tpm2;Col...
6,prerank,Airway Smooth Muscle Cells,0.721296,2.200083,0.0,0.0,0.0,42/72,13.70%,Cald1;Serpinh1;Col1a1;Igfbp7;Col5a2;Col1a2;Tpm...
7,prerank,Osteoblasts,0.687937,2.177178,0.0,0.0,0.0,67/100,25.10%,Cald1;Col1a1;Igfbp7;Sparc;Col5a2;Col1a2;Fstl1;...
8,prerank,Pancreatic Stellate Cells,0.695538,2.164514,0.0,0.0,0.0,52/82,20.80%,Cald1;Col1a1;Sparc;Col5a2;Col1a2;Fstl1;Lox;Col...
11,prerank,Hepatic Stellate Cells,0.684521,2.136636,0.0,0.0,0.0,53/84,22.60%,Cald1;Col1a1;Igfbp7;Sparc;Col5a2;Col1a2;Fstl1;...
12,prerank,Myofibroblasts,0.694664,2.129217,0.0,0.0,0.0,48/73,22.60%,Cald1;Col1a1;Sparc;Col5a2;Col1a2;Tpm1;Fstl1;Tp...
13,prerank,Mesangial Cells,0.673959,2.116045,0.0,0.0,0.0,57/100,20.80%,Cald1;Col1a1;Sparc;Col5a2;Col1a2;Tpm1;Fstl1;Tp...
15,prerank,Chondrocytes,0.669175,2.115248,0.0,0.0,0.0,56/94,22.60%,Cald1;Col1a1;Igfbp7;Sparc;Col5a2;Col1a2;Fstl1;...
16,prerank,Fibroblasts,0.649721,2.104909,0.0,0.0,0.0,86/145,25.15%,Serpinh1;Col1a1;Igfbp7;Sparc;Col5a2;Col1a2;Fst...
19,prerank,Smooth Muscle Cells,0.660003,2.07724,0.0,0.0,0.0,57/92,25.25%,Cald1;Col1a1;Col5a2;Col1a2;Tpm1;Fstl1;Lox;Tpm2...


Still fibroblasts

### Day 4 Hic2 OX

In [30]:
hic2_rank_df = scanpy.get.rank_genes_groups_df(
    adata,
    key="macrostate_rank_genes",
    group="Day 4 Hic2 OX"
)
hic2_rank_df.index = hic2_rank_df["names"]
hic2_rank_df = hic2_rank_df.drop(columns="names")

In [52]:
hic2_rank_df.sort_values("scores", ascending=False).head(20)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Krt20,50.681782,3.986395,0.000000e+00,0.000000e+00
Krtdap,49.634972,3.220684,0.000000e+00,0.000000e+00
Hbb-y,48.800888,3.551641,0.000000e+00,0.000000e+00
Pfn1,42.958733,0.865490,0.000000e+00,0.000000e+00
Ggh,39.460499,1.800804,0.000000e+00,0.000000e+00
Isg15,38.423088,2.758160,0.000000e+00,0.000000e+00
Oasl2,37.218948,2.134203,3.369941e-303,9.628404e-301
Uchl1,35.828896,1.908738,3.920448e-281,8.712106e-279
Bex1,35.728615,1.729572,1.421534e-279,2.843068e-277


In [36]:
hic2_rank_df.loc[numpy.intersect1d(target_tfs, adata.var_names)].sort_values("scores", ascending=False).head(20)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Msc,27.601393,2.815124,1.070682e-167,3.965488e-166
Fosl1,26.082756,1.788012,5.720943e-150,1.658244e-148
Id2,24.631092,1.344160,5.868540e-134,1.414106e-132
Hmga2,20.150198,0.812654,2.680445e-90,4.000663e-89
Hic2,15.447442,0.845011,7.849686e-54,5.793126e-53
Tcf7l2,13.495073,0.549340,1.671893e-41,9.777154e-41
Myc,13.462667,0.575639,2.593794e-41,1.503649e-40
Arid3a,10.510388,0.559938,7.737457e-26,3.076524e-25
Atf5,9.983627,0.333068,1.797723e-23,6.670587e-23


In [33]:
hic2_rank_df.loc[genes_of_interest].sort_values("scores", ascending=False).head(20)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Hic2,15.447442,0.845011,7.849686e-54,5.793126e-53
Myc,13.462667,0.575639,2.593794e-41,1.503649e-40
Zbtb7c,-0.098243,-0.222021,9.217393e-01,9.596453e-01
Tcf7l1,-4.097978,-0.321196,4.167752e-05,7.938576e-05
Klf4,-5.762605,-0.614489,8.282545e-09,1.880260e-08
Ovol1,-7.336517,-1.586220,2.192231e-13,5.893095e-13
Tlx2,-8.593602,-1.362809,8.428459e-18,2.705765e-17
Rest,-17.573996,-0.794881,3.897295e-69,3.916880e-68
Jarid2,-17.899170,-1.343034,1.196946e-71,1.266610e-70


In [ ]:
# Cell types

pre_res = gseapy.prerank(
    rnk=hic2_rank_df[["scores"]], 
    gene_sets="PanglaoDB_Augmented_2021",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)

In [ ]:
# Cell types

pre_res = gseapy.prerank(
    rnk=hic2_rank_df[["scores"]], 
    gene_sets="PanglaoDB_Augmented_2021",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)

2026-03-09 13:51:00,448 [WARNING] Duplicated values found in preranked stats: 0.80% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
0,prerank,Kupffer Cells,0.683866,1.963881,0.0,0.010314,0.009,3/33,5.80%,Lgals9;Hck;Dok2
1,prerank,Osteoclast Precursor Cells,0.643053,1.80056,0.001996,0.059307,0.097,3/29,6.25%,Lgals9;Hck;Ifih1
2,prerank,Neutrophils,0.647191,1.794089,0.0,0.044007,0.108,4/27,5.80%,Mme;Hck;Hdc;Dok2
5,prerank,Alveolar Macrophages,0.595126,1.725795,0.0,0.06859,0.207,4/34,7.20%,Lgals9;Hck;Ifi30;Plet1
7,prerank,Monocytes,0.536578,1.674711,0.003738,0.094685,0.325,8/51,5.80%,Lgals9;Hck;Rgs2;Ifi30;Ifit1;Ly6e;Vcan;Dok2
9,prerank,Noradrenergic Neurons,0.828487,1.565582,0.029915,0.257169,0.666,5/6,14.55%,Ddc;Fetub;Rbp4;Dlk1;Ahsg
11,prerank,Eosinophils,0.541306,1.524645,0.026365,0.312227,0.777,2/25,5.80%,Hck;Dok2
12,prerank,Langerhans Cells,0.537416,1.494656,0.034413,0.349782,0.843,2/27,2.30%,Lgals9;Hck
13,prerank,Olfactory Epithelial Cells,0.558734,1.468446,0.044402,0.381054,0.89,8/22,11.90%,Dusp14;Rnf128;Pfn2;Fam81a;Abhd16a;Fetub;Atf5;H...
14,prerank,Decidual Cells,0.771674,1.455336,0.054622,0.378017,0.902,1/6,5.80%,Dok2


Interestingly, the Hic2 OX cells look like immune cells

In [53]:
# MSig Hallmarks in Hic2 OX pathway

pre_res = gseapy.prerank(
    rnk=hic2_rank_df[["scores"]], 
    gene_sets="MSigDB_Hallmark_2020",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)

2026-03-09 13:58:02,613 [WARNING] Duplicated values found in preranked stats: 0.80% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
0,prerank,Interferon Alpha Response,0.689688,2.044943,0.0,0.0,0.0,24/38,21.20%,Isg15;Ifi35;B2m;Ifi30;Rtp4;Rsad2;Ly6e;Lgals3bp...
1,prerank,Interferon Gamma Response,0.602988,1.968047,0.0,0.0,0.0,22/64,10.35%,Isg15;Xaf1;Zbp1;Psmb10;Ifi35;B2m;Ifi30;Rtp4;Rs...
2,prerank,E2F Targets,0.60779,1.703624,0.00611,0.023674,0.077,15/28,18.70%,Ccne1;Cdkn2a;Birc5;Rrm2;Cdkn1a;Pttg1;Myc;Aurka...
3,prerank,Fatty Acid Metabolism,0.617043,1.540223,0.037657,0.116965,0.406,6/17,16.15%,Mgll;Hsp90aa1;Cbr3;Eno3;Hmgcs1;Glul
4,prerank,mTORC1 Signaling,0.506861,1.499783,0.027613,0.129793,0.502,14/36,12.80%,Ppa1;Psph;Cth;Ifi30;Nupr1;Rrm2;Cdkn1a;Ctsc;Ifr...
5,prerank,Myc Targets V1,0.796374,1.448456,0.073922,0.157284,0.622,3/5,17.75%,Ifrd1;Myc;Cdc20
9,prerank,Inflammatory Response,0.416672,1.353401,0.055028,0.259104,0.846,19/60,17.90%,Rtp4;Ly6e;Cdkn1a;Atp2b1;Ccl7;Emp3;Gch1;Myc;Pla...
10,prerank,Cholesterol Homeostasis,0.503217,1.345944,0.099029,0.236814,0.858,10/24,15.45%,Clu;Anxa5;Fabp5;Lgals3;S100a11;Plaur;Atf5;Cd9;...
11,prerank,Complement,0.406522,1.303914,0.068898,0.273041,0.909,10/58,10.75%,Pfn1;Clu;Calm1;Anxa5;Lgals3;Pla2g7;Ctsc;Hspa5;...
14,prerank,PI3K/AKT/mTOR Signaling,0.511857,1.25762,0.162602,0.319511,0.946,6/16,13.55%,Pfn1;Cfl1;Cdkn1a;Pla2g12a;Cdk1;Stat2


In [54]:
# GO biological process in Hic2 OX pathway

pre_res = gseapy.prerank(
    rnk=hic2_rank_df[["scores"]], 
    gene_sets="GO_Biological_Process_2025",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)[["Term", "NES", "NOM p-val", "FWER p-val", "FDR q-val"]]

2026-03-09 14:00:00,485 [WARNING] Duplicated values found in preranked stats: 0.80% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Term,NES,NOM p-val,FWER p-val,FDR q-val
0,Negative Regulation of Viral Genome Replicatio...,1.743962,0.008511,0.747,1.0
1,Mitotic Cell Cycle Phase Transition (GO:0044772),1.741767,0.006579,0.751,0.94428
3,Regulation of Cell Cycle Process (GO:0010564),1.70615,0.006224,0.883,1.0
4,Regulation of Cytokine Production (GO:0001817),1.693619,0.005871,0.92,0.906386
6,Protein Localization to Nucleus (GO:0034504),1.682094,0.004292,0.94,0.84505
7,Positive Regulation of Pattern Recognition Rec...,1.679265,0.013304,0.949,0.732609
8,Negative Regulation of Leukocyte Degranulation...,1.67372,0.002179,0.965,0.674485
9,Negative Regulation of Viral Process (GO:0048525),1.67054,0.020284,0.97,0.614182
10,Regulation of Interleukin-10 Production (GO:00...,1.669307,0.012821,0.972,0.555145
11,Regulation of Viral Genome Replication (GO:004...,1.663897,0.018367,0.974,0.540584


### Day 9 Hic2 OX

In [37]:
late_rank_df = scanpy.get.rank_genes_groups_df(
    adata,
    key="macrostate_rank_genes",
    group="Day 9 Hic2 OX"
)
late_rank_df.index = late_rank_df["names"]
late_rank_df = late_rank_df.drop(columns="names")

In [41]:
late_rank_df.loc[numpy.intersect1d(target_tfs, adata.var_names)].sort_values("scores", ascending=False).head(20)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Tcf7l2,37.064835,1.508311,1.036050e-300,1.883727e-298
Jarid2,30.248821,1.286976,5.406728e-201,2.922556e-199
Tcf7l1,23.495508,1.071344,4.533657e-122,9.158904e-121
Zfp42,22.298857,0.912690,3.790599e-110,6.535515e-109
Fos,20.607384,1.226833,2.356247e-94,3.390283e-93
Tfcp2l1,18.570190,0.933905,5.600398e-77,6.087390e-76
Tfap2c,18.173567,1.414879,8.358384e-74,8.844851e-73
Sall1,15.699179,0.991098,1.532119e-55,1.174037e-54
Hic2,11.966811,0.668471,5.302742e-33,2.537197e-32


In [40]:
# Cell types

pre_res = gseapy.prerank(
    rnk=late_rank_df[["scores"]], 
    gene_sets="PanglaoDB_Augmented_2021",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)

2026-03-09 13:43:14,698 [WARNING] Duplicated values found in preranked stats: 0.30% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
0,prerank,Kupffer Cells,0.785404,2.070219,0.0,0.0,0.0,5/33,10.50%,Pltp;Lgals9;Hck;Dok2;Ncf2
6,prerank,Osteoclast Precursor Cells,0.693428,1.813739,0.0,0.018252,0.048,6/29,20.60%,Lgals9;Hck;Cd68;Ncf2;Laptm5;Samd9l
11,prerank,Pluripotent Stem Cells,0.684123,1.762792,0.0,0.025552,0.095,19/25,22.70%,Gdf3;Terf1;Trim71;Sall4;Dnmt3b;Msh6;Hells;Mcm3...
18,prerank,Macrophages,0.551475,1.65649,0.001355,0.081403,0.315,9/59,10.50%,Pltp;Ucp2;Lgals9;Hck;Cd68;Dusp5;Cd200;Dok2;Ncf2
19,prerank,Langerhans Cells,0.639908,1.653216,0.004673,0.067458,0.323,5/27,22.55%,Lgals9;Hck;Ncf2;Laptm5;Snx20
21,prerank,Red Pulp Macrophages,0.644078,1.645733,0.001534,0.063394,0.356,4/26,19.55%,Hck;Ncf2;Hmox1;Slc7a7
22,prerank,Microglia,0.575776,1.640523,0.004348,0.058093,0.368,8/46,11.85%,Ucp2;Lgals9;Hck;Fos;Sall1;Dok2;Ncf2;Pecam1
25,prerank,Eosinophils,0.618619,1.580054,0.01214,0.106499,0.565,7/25,28.25%,Hck;Dok2;Laptm5;Ltc4s;Snx20;Pglyrp1;Arhgap30
26,prerank,Alveolar Macrophages,0.593271,1.579414,0.007278,0.095396,0.569,7/34,18.20%,Plet1;Lgals9;Hck;Cd68;Ncf2;Gngt2;Laptm5
28,prerank,Gamma Delta T Cells,0.709927,1.545743,0.026362,0.123017,0.687,9/12,19.90%,Hmgb2;Top2a;Cenpa;Cenpf;Aurkb;Birc5;Mki67;Nusa...


Still like immune cells, but also pluripotent stem cells.

### Stem cells

In [43]:
mesc_rank_df = scanpy.get.rank_genes_groups_df(
    adata,
    key="macrostate_rank_genes",
    group="mESC"
)
mesc_rank_df.index = mesc_rank_df["names"]
mesc_rank_df = mesc_rank_df.drop(columns="names")

In [47]:
mesc_rank_df.loc[genes_of_interest].sort_values("scores", ascending=False).head(20)

,scores,logfoldchanges,pvals,pvals_adj
names,,,,
Zfp42,67.341812,4.388525,0.000000e+00,0.000000e+00
Rest,62.072388,2.318409,0.000000e+00,0.000000e+00
Jarid2,60.975414,3.011444,0.000000e+00,0.000000e+00
Klf4,33.226761,1.959148,4.422481e-242,2.862447e-241
Tcf7l1,25.351522,1.151651,8.645614e-142,3.800270e-141
Cdh1,25.069399,1.208922,1.072874e-138,4.644478e-138
Hic2,16.042459,0.843960,6.454696e-58,1.705336e-57
Ovol1,-12.893654,-3.375580,4.887225e-38,1.117080e-37
Tlx2,-19.955168,-4.166796,1.351705e-88,4.388653e-88


In [46]:
# Cell types

pre_res = gseapy.prerank(
    rnk=mesc_rank_df[["scores"]], 
    gene_sets="PanglaoDB_Augmented_2021",
    threads=32,
    min_size=5,
    max_size=1000
)
pre_res.res2d.sort_values("NES", ascending=False).head(20)

2026-03-09 13:48:47,693 [WARNING] Duplicated values found in preranked stats: 0.30% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
0,prerank,Epiblast Cells,0.79995,2.347374,0.0,0.0,0.0,21/33,14.95%,Tdgf1;Phc1;Sall4;Nanog;Fgf4;Trim71;Cecr2;Ina;M...
1,prerank,Pluripotent Stem Cells,0.854374,2.344033,0.0,0.0,0.0,24/25,14.95%,Tdgf1;Nasp;Msh6;Hells;Mcm3;Sall4;Dppa2;Nanog;F...
3,prerank,Oxyphil Cells,0.758383,1.922956,0.0,0.002552,0.011,5/17,8.55%,Esrrb;Cecr2;Tfdp2;Lrrc2;Ppp1r1a
12,prerank,Gamma Delta T Cells,0.785635,1.798505,0.002141,0.01601,0.077,9/12,14.10%,Ifitm1;Hmgb2;Top2a;Cenpf;Nusap1;Aspm;Mki67;Aur...
28,prerank,Reticulocytes,0.787768,1.581462,0.035556,0.164138,0.589,4/7,10.00%,Mcm3;Dhx16;Incenp;Folr1
34,prerank,Embryonic Stem Cells,0.449199,1.521405,0.015054,0.226925,0.759,31/66,17.45%,Tdgf1;Zfp42;Utf1;Dppa4;Fbxo15;Sall4;Enah;Dppa2...
37,prerank,Proximal Tubule Cells,0.614215,1.481963,0.043197,0.263719,0.853,6/14,18.70%,Apoe;Alpl;Folr1;Napsa;Ass1;Lrp2
44,prerank,Glycinergic Neurons,0.610196,1.405129,0.087398,0.39303,0.948,5/12,17.95%,Ina;Aplp1;Ckb;Gng3;Elavl3
45,prerank,Adrenergic Neurons,0.617804,1.401959,0.073684,0.355315,0.949,4/11,14.75%,Fxyd6;Mycn;Pcsk1n;Palm3
47,prerank,Retinal Ganglion Cells,0.509504,1.383505,0.071279,0.362315,0.97,3/24,6.35%,Rbpms2;Ina;Aplp1


# Find which transcription factors are upregulated in the dead end with Decoupler

Databases available in Decoupler: https://omnipathdb.org/info.html

## CollecTRI

In [80]:
# Load CollecTRI database

collectri_df = decoupler.op.collectri(organism="mouse")

In [141]:
# TF activity in dead end CollecTRI

data = dead_end_rank_df[["scores"]].T.dropna(axis=1)
tf_acts, tf_padj = decoupler.mt.ulm(data=data, net=collectri_df)
activities_df = pandas.DataFrame(data={"Activity": tf_acts.transpose()["scores"], "Adjusted p-value": tf_padj.transpose()["scores"]}, index=tf_acts.columns)
activities_df.sort_values("Activity", ascending=False).head(30)

,Activity,Adjusted p-value
Trp73,4.029077,0.016108
Ppara,3.984019,0.016108
Atf4,3.839768,0.019429
Atf6,3.703987,0.025015
Klf4,3.611301,0.026601
Arnt,3.387582,0.047047
Klf6,3.351070,0.047047
Pparg,3.284603,0.048020
Trp53,3.282625,0.048020
Spdef,3.254378,0.048211


In [100]:
# Most downregulated in dead end

activities_df.sort_values("Activity", ascending=True).head(25)

,Activity,Adjusted p-value
Thrb,-3.934584,0.013187
Ovol1,-3.554015,0.033014
Atf3,-3.525898,0.033014
Smad3,-3.192313,0.093985
Smad4,-3.086192,0.117917
Tcf21,-2.984713,0.146517
Pgr,-2.727191,0.268868
Hbp1,-2.688656,0.276686
Tfcp2,-2.589001,0.286822
Zfp42,-2.580047,0.286822


In [143]:
# TF activity in day 4 control, CollecTRI

data = control_rank_df[["scores"]].T.dropna(axis=1)
tf_acts, tf_padj = decoupler.mt.ulm(data=data, net=collectri_df)
activities_df = pandas.DataFrame(data={"Activity": tf_acts.transpose()["scores"], "Adjusted p-value": tf_padj.transpose()["scores"]}, index=tf_acts.columns)
print("Top")
IPython.display.display(activities_df.sort_values("Activity", ascending=False).head(30))
print("Bottom")
IPython.display.display(activities_df.sort_values("Activity", ascending=True).head(30))

Top


,Activity,Adjusted p-value
Klf8,4.716857,0.001175
Mybl2,4.475703,0.001846
Relb,3.659164,0.029789
Kmt2a,2.742200,0.268868
Sox17,2.588997,0.286822
Kdm5b,2.523081,0.286822
Tcf4,2.364670,0.340264
Meis1,2.329665,0.340264
Hnf4a,2.295128,0.340264
Isl1,2.252731,0.340264


Bottom


,Activity,Adjusted p-value
Thrb,-3.934584,0.013187
Ovol1,-3.554015,0.033014
Atf3,-3.525898,0.033014
Smad3,-3.192313,0.093985
Smad4,-3.086192,0.117917
Tcf21,-2.984713,0.146517
Pgr,-2.727191,0.268868
Hbp1,-2.688656,0.276686
Tfcp2,-2.589001,0.286822
Zfp42,-2.580047,0.286822


Dorothea is an alternative, but it has fewer TFs and targets.

## Progeny database

In [ ]:
# Load database

progeny_df = decoupler.op.progeny(organism="mouse")

In [122]:
# Progeny pathways in dead end

data = dead_end_rank_df[["scores"]].T.dropna(axis=1)
tf_acts, tf_padj = decoupler.mt.ulm(data=data, net=progeny_df)
activities_df = pandas.DataFrame(data={"Activity": tf_acts.transpose()["scores"], "Adjusted p-value": tf_padj.transpose()["scores"]}, index=tf_acts.columns)

IPython.display.display(activities_df.sort_values("Activity", ascending=False))

,Activity,Adjusted p-value
p53,11.175544,5.049649e-27
Hypoxia,4.085607,2.132595e-04
NFkB,2.682470,2.063120e-02
VEGF,2.560395,2.327722e-02
Trail,2.525243,2.327722e-02
TNFa,2.147510,5.577717e-02
Androgen,1.364986,2.194324e-01
EGFR,1.102954,3.152097e-01
MAPK,0.268889,7.880430e-01
WNT,-0.282407,7.880430e-01


## Hallmark database

In [123]:
# Load database

hallmark_df = decoupler.op.hallmark(organism="mouse")

In [126]:
# Hallmark pathways in dead end

data = dead_end_rank_df[["scores"]].T.dropna(axis=1)
tf_acts, tf_padj = decoupler.mt.ulm(data=data, net=hallmark_df)
activities_df = pandas.DataFrame(data={"Activity": tf_acts.transpose()["scores"], "Adjusted p-value": tf_padj.transpose()["scores"]}, index=tf_acts.columns)

print("Top")
IPython.display.display(activities_df.sort_values("Activity", ascending=False).head(20))
print("Bottom")
IPython.display.display(activities_df.sort_values("Activity", ascending=True).head(20))

Top


,Activity,Adjusted p-value
P53_PATHWAY,7.402222,9.618454e-12
TNFA_SIGNALING_VIA_NFKB,5.438254,5.919235e-07
MTORC1_SIGNALING,3.577776,2.287099e-03
HYPOXIA,3.512976,2.466146e-03
ESTROGEN_RESPONSE_EARLY,3.297621,4.537892e-03
UV_RESPONSE_UP,3.290162,4.537892e-03
ANDROGEN_RESPONSE,2.225769,8.539350e-02
MYC_TARGETS_V1,2.137684,9.539345e-02
UNFOLDED_PROTEIN_RESPONSE,2.104697,9.648170e-02
ESTROGEN_RESPONSE_LATE,2.044442,1.005498e-01


Bottom


,Activity,Adjusted p-value
EPITHELIAL_MESENCHYMAL_TRANSITION,-7.094219,4.406731e-11
G2M_CHECKPOINT,-6.797039,2.294860e-10
E2F_TARGETS,-5.995401,2.943103e-08
MITOTIC_SPINDLE,-5.051003,3.915482e-06
INTERFERON_ALPHA_RESPONSE,-3.564292,2.287099e-03
OXIDATIVE_PHOSPHORYLATION,-2.650596,3.144966e-02
UV_RESPONSE_DN,-2.640461,3.144966e-02
ANGIOGENESIS,-2.391771,5.900746e-02
APICAL_JUNCTION,-2.132399,9.539345e-02
SPERMATOGENESIS,-2.059392,1.005498e-01


In [128]:
# Hallmark pathways in day 4 control

data = control_rank_df[["scores"]].T.dropna(axis=1)
tf_acts, tf_padj = decoupler.mt.ulm(data=data, net=hallmark_df)
activities_df = pandas.DataFrame(data={"Activity": tf_acts.transpose()["scores"], "Adjusted p-value": tf_padj.transpose()["scores"]}, index=tf_acts.columns)

print("Top")
IPython.display.display(activities_df.sort_values("Activity", ascending=False).head(20))
print("Bottom")
IPython.display.display(activities_df.sort_values("Activity", ascending=True).head(20))

Top


,Activity,Adjusted p-value
ESTROGEN_RESPONSE_LATE,3.900823,0.002426
ESTROGEN_RESPONSE_EARLY,2.895034,0.062595
MYC_TARGETS_V1,2.642994,0.081162
MTORC1_SIGNALING,2.541076,0.088094
G2M_CHECKPOINT,2.484186,0.088094
E2F_TARGETS,2.443250,0.088094
P53_PATHWAY,2.225636,0.128134
WNT_BETA_CATENIN_SIGNALING,1.998669,0.203066
NOTCH_SIGNALING,1.928981,0.203066
ANDROGEN_RESPONSE,1.794024,0.255362


Bottom


,Activity,Adjusted p-value
EPITHELIAL_MESENCHYMAL_TRANSITION,-5.090008,0.000019
UV_RESPONSE_DN,-2.691453,0.081162
INTERFERON_ALPHA_RESPONSE,-2.406862,0.088094
HYPOXIA,-1.957955,0.203066
APICAL_JUNCTION,-1.690011,0.279244
ADIPOGENESIS,-1.492006,0.369829
OXIDATIVE_PHOSPHORYLATION,-1.407434,0.411224
BILE_ACID_METABOLISM,-1.347908,0.435710
DNA_REPAIR,-1.319693,0.436541
APOPTOSIS,-1.283101,0.444576


## Trrust database

In [131]:
# Load database

trrust_df = decoupler.op.resource("TRRUST", organism="human")

AssertionError: name must be one of these: {'ComPPI', 'SignaLink_function', 'PanglaoDB', 'TFcensus', 'Kirouac2010', 'KEGG-PC', 'GO_Intercell', 'Guide2Pharma', 'connectomeDB2020', 'Matrisome', 'CancerGeneCensus', 'Almen2009', 'MSigDB', 'Surfaceome', 'HPA_tissue', 'CellCall', 'Wang', 'InterPro', 'Phosphatome', 'CellChatDB_complex', 'MatrixDB', 'Membranome', 'LRdb', 'MCAM', 'Vesiclepedia', 'iTALK', 'CPAD', 'KEGG', 'SIGNOR', 'Adhesome', 'Baccin2019', 'Integrins', 'DGIdb', 'HPA_subcellular', 'HPMR', 'NetPath', 'Cellinker', 'Ramilowski_location', 'TopDB', 'GPCRdb', 'ICELLNET_complex', 'PROGENy', 'OPM', 'scConnect_complex', 'HGNC', 'TCDB', 'Zhong2015', 'kinase.com', 'ICELLNET', 'CellPhoneDB', 'EMBRACE', 'DisGeNet', 'LOCATE', 'talklr', 'HumanCellMap', 'CellTypist', 'CancerDrugsDB', 'UniProt_family', 'CSPA_celltype', 'CellPhoneDB_complex', 'CytoSig', 'CORUM_Funcat', 'SignaLink_pathway', 'CellTalkDB', 'IntOGen', 'Cellinker_complex', 'Exocarta', 'Phobius', 'CellCellInteractions', 'CellChatDB', 'L